Resource
-

In [1]:
!pip install wandb==0.22.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 52.9 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.25.1
    Uninstalling wandb-0.25.1:
      Successfully uninstalled wandb-0.25.1


### Install kaleido for export pdf

In [2]:
!pip install plotly==6.4.0
!pip install kaleido==1.2.0
!pip install choreographer==1.2.1
!pip install pypdf
!apt update
!apt-get install libnss3 libatk-bridge2.0-0 libcups2 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libxkbcommon0 libpango-1.0-0 libcairo2 libasound2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 25.5 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: choreographer
    Found existing installation: choreographer 1.4.0
    Uninstalling choreographer-1.4.0:
      Successfully uninstalled choreographer-1.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 15.6 MB/s eta 0:00:00
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease

In [3]:
import plotly.io as pio

pio.get_chrome()

PosixPath('/usr/local/lib/python3.12/dist-packages/choreographer/cli/browser_exe/chrome-linux64/chrome')

# Login

In [1]:
from google.colab import userdata
WANDB_KEY = userdata.get('WANDB_TOKEN')

In [2]:
import wandb
print(wandb.__version__)
wandb.login(key=WANDB_KEY)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

0.22.0


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rofifpriyo (rofifpriyo-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Parameter

In [3]:
import wandb
from wandb.apis.public import Api
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.subplots as ps
import plotly.graph_objects as go

ENTITY = wandb.apis.PublicApi().default_entity
PROJECT = "calibration-on-quantized-multilingual"

# Function

## Deliverance to CSV

In [5]:
import wandb
from wandb.apis.public import Api
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.subplots as ps
import plotly.graph_objects as go

def deliver_to_csv(eval_languages, run, base_models, eval_dataset, bits, custom_title="", write_csv=""):
  # Ensure that bits is a list not list of list
  if isinstance(bits, list):
    if len(bits) > 0 and isinstance(bits[0], list):
      bits = bits[max(enumerate([len(bit) for bit in bits]), key=lambda x: x[1])[0]]

  language_sort = {
      # Flores Plus
      "eng_Latn": 0, "fra_Latn": 1, "hin_Deva": 2, "npi_Deva": 3, "tam_Taml": 6, "tel_Telu": 7, "ind_Latn": 8, "zsm_Latn": 9, "swh_Latn": 10, "yor_Latn": 11, "cmn_Hans": 12, "jpn_Jpan": 13, "kor_Hang": 14, "arb_Arab" : 15,\
      # Wikipedia
      "en": 0, "fr": 1, "hi": 2, "ne": 3, "ta": 6, "te": 7, "id": 8, "ms": 9, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,\
      # INCLUDE
      "french": 1, "hindi": 2, "nepali": 3, "albanian": 4, "tamil": 6, "telugu": 7, "indonesian": 8, "malay": 9, "chinese": 12, "japanese": 13, "korean": 14, "arabic": 15,\
      "dutch": 0.2, "german": 0.4, "tagalog": 8.5, "malayalam": 7.5,\
      # Global MMLU Lite
      "en": 0, "fr": 1, "hi": 2, "bn": 5, "id": 8, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,
      # Multi Blimp
      "eng": 0, "tam": 6, "idn": 8, "swa": 10, "zhoblimp": 12,
      # Belebele
      "eng_Latn": 0, "nld_Latn": 0.2, "deu_Latn": 0.4, "mal_Mlym": 5.8, "tam_Taml": 6, "tel_Telu": 7, "ind_Latn": 8, "tgl_Latn": 8.5, "zsm_Latn": 9, "swh_Latn": 10, "yor_Latn": 11, "lug_Latn": 10.5, "bod_Tibt": 12.2, "mya_Mymr": 12.4, "zho_Hans": 12,
      # Massive
      'en-US': 0, 'id-ID': 8, 'ta-IN': 6, 'sw-KE': 10, 'zh-CN': 12,
      # MGSM
      "en": 0, "ta": 6, "sw": 10, "zh": 12,
  }
  if eval_dataset.lower() == 'floresplus':
    flores_plus_add = ["afr_Latn", "dan_Latn", "deu_Latn", "eng_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 0 + index / 100
    flores_plus_add = ["kan_Knda", "mal_Mlym", "tam_Taml", "tel_Telu"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 6 + index / 100
    flores_plus_add = ["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "ind_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 8 + index / 100
    flores_plus_add = ["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "swh_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 10 + index / 100
    flores_plus_add = ["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant", "cmn_Hans"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 12 + index / 100

  custom_sort = {'Unquantized': 0, 'English': 1, 'Tamil': 2, 'Indonesian': 3, 'Swahili': 4, 'Chinese': 5,\
                 'Unquantized': 0, 'gptq': 1, 'slimllm': 2, 'tacq': 3,\
                 "Qwen/Qwen3-8B": 0, "CohereLabs/aya-expanse-8b": 1, "meta-llama/Llama-3.1-8B-Instruct": 2, "Qwen/Qwen3-14B": 3,\
                 "32-bit": 0, "2-bit": 1, "3-bit": 2, "4-bit": 3}

  calib_label_list = ["EN", "TA", "ID", "SW", "ZH"]
  calib_label = {"English": "EN", "Tamil": "TA", "Indonesian": "ID", "Swahili": "SW", "Chinese": "ZH"}
  calib_label_reversed = {v: k for k, v in calib_label.items()}
  base_model_label = {"Qwen/Qwen3-8B": "Qwen3", "meta-llama/Llama-3.1-8B-Instruct": "Llama", "CohereLabs/aya-expanse-8b": "Aya"}
  eval_label = {
      # Global PIQA
      "eng_latn": "EN", "tam_taml": "TA", "ind_latn": "ID", "swh_latn": "SW", "cmn_hans": "ZH",\
      # INCLUDE
      "french": "FR", "hindi": "HI", "nepali": "NE", "albanian": "SQ", "tamil": "TA", "telugu": "TE", "indonesian": "ID", "malay": "MS", "chinese": "ZH", "japanese": "JA", "korean": "KO", "arabic": "AR",\
      "dutch": "NL", "german": "DE", "tagalog": "TL", "malayalam": "ML",\
      "english": "EN",\
      # Global MMLU Lite
      "en": "EN", "fr": "FR", "hi": "HI", "bn": "BN", "id": "ID", "sw": "SW", "yo": "YO", "zh": "ZH", "ja": "JA", "ko": "KO", "ar": "AR",
      # Multi Blimp
      "eng": "EN", "tam": "TA", "idn": "ID", "swa": "SW", "zhoblimp": "ZH",
      # Belebele
      "eng_Latn": "EN", "nld_Latn": "NL", "deu_Latn": "DE", "mal_Mlym": "ML", "tam_Taml": "TA", "tel_Telu": "TE", "ind_Latn": "ID", "tgl_Latn": "TL", "zsm_Latn": "MS", "swh_Latn": "SW", "yor_Latn": "YO", "lug_Latn": "LG", "bod_Tibt": "BO", "mya_Mymr": "MY", "zho_Hans": "ZH",
      # Massive
      'en-US': "EN", 'id-ID': "ID", 'ta-IN': "TA", 'sw-KE': "SW", 'zh-CN': "ZH",
      # MGSM
      "en": "EN", "ta": "TA", "sw": "SW", "zh": "ZH",
  }
  family_mapper = {
      # INCLUDE
      "tamil": "TA", "telugu": "TA", "indonesian": "ID", "malay": "ID", "chinese": "ZH",\
      "dutch": "EN", "german": "EN", "tagalog": "ID", "malayalam": "TA",\
      "english": "EN",\
      # Non-existance, but for the sake of labeling
      # Global MMLU Lite
      "en": "EN", "de": "EN", "my": "ID", "id": "ID", "sw": "SW", "yo": "SW", "zh": "ZH",
  }


  eval_languages = list(sorted(eval_languages, key=lambda x: language_sort.get(x, 100)))
  columns = ["Base Model", "Quantization Technique", "Bit Width", "Calibration Language", "Random Seed"]
  columns.extend(eval_languages)
  if "floresplus" in eval_dataset.lower():
    columns.extend([f"{eval}_nancount" for eval in eval_languages])

  data_acc = []

  # Iterate over the runs and print details
  for run in runs:
    config = run.config
    summary = run.summary

    if config.get("calibration_language", None) == "WuuChinese":
      continue

    run_data_acc = []
    run_data_acc.extend([run.config["base_model"], config["quantization_technique"], config["bit_width"], config.get("calibration_language", None), config.get("random_seed", 1234)])
    run_data_nancount = []
    for lang in eval_languages:
      if "mgsm" in eval_dataset.lower():
          strict = run.summary.get(f"{eval_dataset.lower()}_strict_{lang}", None)
          flexible = run.summary.get(f"{eval_dataset.lower()}_flexible_{lang}", None)
          run_data_acc.append(max(strict, flexible))
      elif "floresplus" in eval_dataset.lower():
          run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_gmean_perplexity_{lang}",
                                              run.summary.get(f"{eval_dataset.lower()}_word_acc_{lang}", None)
                                              ))
          run_data_nancount.append(run.summary.get(f"{eval_dataset.lower()}_nancount_{lang}", 0))
      else:
        run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_acc_{lang}", None))

    if "floresplus" in eval_dataset:
      run_data_acc.extend(run_data_nancount)
    data_acc.append(run_data_acc)

  df_acc_chief = pd.DataFrame(data_acc, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language', "Bit Width"], key=lambda x: x.map(custom_sort))

  if eval_dataset.lower() == "include" and not compile and 'telugu' not in eval_languages:
    df_acc_chief.insert(5, 'english', (df_acc['german'] + df_acc['dutch']) / (2))
    df_acc_chief.drop(["german", "dutch"], axis=1, inplace=True)

    replace_index = min(eval_languages.index("german"), eval_languages.index("dutch"))
    eval_languages.insert(replace_index, "english")
    eval_languages.remove("german")
    eval_languages.remove("dutch")

  display(df_acc_chief)
  df_acc_chief.to_csv(write_csv if write_csv else f"{eval_dataset}-results.csv", index=False)
  print(f"Finish processing {write_csv if write_csv else f'{eval_dataset}-results.csv'} with shape: {df_acc_chief.shape}")

In [6]:
# Initialize the W&B API
api = Api()

# Parameters
list_eval_languages = []
list_runs = []
list_base_models = []
list_eval_dataset = []
list_bits = []
list_save_names = []

# FloresPlus
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "cmn_Hans"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen3-14B"]
eval_dataset = "floresplus"
cross_lingual = "-"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append(f"{eval_dataset}{cross_lingual}results.csv")

# FloresPlus Germanic
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "cmn_Hans"]
eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "floresplus"
cross_lingual = "-germanic-"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append(f"{eval_dataset}{cross_lingual}results.csv")

# FloresPlus Tamil Dravidian
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "cmn_Hans"]
eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "floresplus"
cross_lingual = "-dravidian-"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append(f"{eval_dataset}{cross_lingual}results.csv")

# FloresPlus Indonesia Malayo Polynesian
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "cmn_Hans"]
eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "floresplus"
cross_lingual = "-malayopolynesian-"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append(f"{eval_dataset}{cross_lingual}results.csv")

# FloresPlus Swahili Niger-Congo
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "cmn_Hans"]
eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "floresplus"
cross_lingual = "-nigercongo-"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append(f"{eval_dataset}{cross_lingual}results.csv")

# FloresPlus Chinese Sino-Tibetan
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "cmn_Hans"]
eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "floresplus"
cross_lingual = "-sinotibetan-"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}-evaluate"},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append(f"{eval_dataset}{cross_lingual}results.csv")

# MultiBlimp
eval_languages = ["eng", "tam", "idn", "swa", "zhoblimp"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen3-14B"]
eval_dataset = "MultiBlimp"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append("")

# Massive Scenario
eval_languages = ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN']
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "Qwen/Qwen3-14B"]
eval_dataset = "Massive"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append("")

# XStoryCloze
eval_languages = ["en", "id", "sw", "zh"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "Qwen/Qwen3-14B"]
eval_dataset = "XStoryCloze"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append("")

# # m ARC
# eval_languages = ["en", "id", "zh"]
# num_samples = 128
# base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
# eval_dataset = "ARC"
# bits = ["2-bit", "3-bit", "4-bit"]

# runs = api.runs(path=f"{ENTITY}/{PROJECT}",
#                 filters={"$or": [
#                         { "$and": [
#                           {"config.num_calibration_samples": num_samples},
#                           {"config.evaluation_dataset": eval_dataset.lower()},
#                           {"config.enable_thinking": False},
#                           {"config.output_type": "multiple_choice"},
#                           {"$not": {"config.calibration_language": "WuuChinese"}},
#                           {"$not": {"config.calibration_language": "Swahili"}},
#                           {"$not": {"config.calibration_language": "Tamil"}},
#                           # {"$not": {"config.bit_width": "2-bit"}},
#                         ]
#                         },
#                         { "$and" : [
#                             {"config.quantization_technique": "Unquantized"},
#                           {"config.evaluation_dataset": eval_dataset.lower()},
#                             {"config.enable_thinking": False},
#                         ]}
#                     ]}
#                 )

# list_eval_languages.append(eval_languages)
# list_runs.append(runs)
# list_base_models.append(base_models)
# list_eval_dataset.append(eval_dataset)
# list_bits.append(bits)

# Belebele
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen3-14B"]
eval_dataset = "Belebele"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append("")

# Global MMLU Lite
eval_languages = ["en", "zh", "id", "sw"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen3-14B"]
eval_dataset = "GlobalMMLULite"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B-Base"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B-Instruct-2507"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B-Base"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B-Instruct-2507"}},

                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append("")

# INCLUDE
eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Tamil"]
eval_languages = [eval_lang.lower() for eval_lang in eval_languages]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen3-14B"]
eval_dataset = "INCLUDE"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append("")

# MGSM
eval_languages = ["en", "ta", "sw", "zh"]
eval_languages = [eval_lang.lower() for eval_lang in eval_languages]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen3-14B"]
eval_dataset = "MGSM"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          # {"config.quantization_technique": quantization_technique.lower()},
                          # {"config.bit_width": f"{bit}-bit"},
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                          # {"config.base_model": base_model},
                          {"config.enable_thinking": True},
                          # {"config.output_type": "generate_until"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"created_at": {"$gt": "2026-07-01"}},
                        ]
                        },
                        { "$and": [
                          {"config.quantization_technique": "sinq"},
                          # {"config.bit_width": f"{bit}-bit"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          # {"config.base_model": base_model},
                          {"config.enable_thinking": True},
                          # {"config.output_type": "generate_until"},
                          {"created_at": {"$gt": "2026-07-01"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": True},
                            {"created_at": {"$gt": "2026-07-01"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_save_names.append("")

# Run the function
print("mean")
for eval_languages, runs, base_models, eval_dataset, bits, save_names in zip(list_eval_languages, list_runs, list_base_models, list_eval_dataset, list_bits, list_save_names):
  deliver_to_csv(eval_languages, runs, base_models, eval_dataset, bits, write_csv=save_names)

mean


,Base Model,Quantization Technique,Bit Width,Calibration Language,Random Seed,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans,eng_Latn_nancount,tam_Taml_nancount,ind_Latn_nancount,swh_Latn_nancount,cmn_Hans_nancount
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,1234,33.428698,2.741605,16.699008,64.534645,46.980806,0,0,0,0,0
1,Qwen/Qwen3-8B,gptq,2-bit,English,1234,13062.876244,48256.105485,39854.463930,38231.523295,34286.236528,319,984,939,882,998
135,Qwen/Qwen3-8B,gptq,2-bit,English,1000,12451.018302,40756.553633,38410.667460,41726.485947,39319.365601,295,674,860,918,988
136,Qwen/Qwen3-8B,gptq,2-bit,English,2000,12969.810741,48809.762083,39445.203775,41236.898022,44868.795841,373,925,923,943,996
6,Qwen/Qwen3-8B,gptq,3-bit,English,1234,40.188512,4.329966,24.304464,160.417519,62.901249,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230,meta-llama/Llama-3.1-8B-Instruct,sinq,2-bit,None,1234,8261.823109,47119.318143,77773.452596,88993.941008,32212.602005,0,0,0,0,0
229,meta-llama/Llama-3.1-8B-Instruct,sinq,3-bit,None,1234,50.553287,3.214247,36.127868,54.643300,50.598038,0,0,0,0,0
231,meta-llama/Llama-3.1-8B-Instruct,sinq,4-bit,None,1234,43.628489,2.412915,25.845251,32.918649,39.595530,0,0,0,0,0
233,Qwen/Qwen3-14B,sinq,2-bit,None,1234,93245.028692,3703.413056,21716.156280,189746.386430,134401.759076,0,0,0,0,0


Finish processing floresplus-results.csv with shape: (234, 15)


,Base Model,Quantization Technique,Bit Width,Calibration Language,Random Seed,afr_Latn,dan_Latn,deu_Latn,eng_Latn,fao_Latn,...,nld_Latn_nancount,nno_Latn_nancount,nob_Latn_nancount,swe_Latn_nancount,tpi_Latn_nancount,ydd_Hebr_nancount,tam_Taml_nancount,ind_Latn_nancount,swh_Latn_nancount,cmn_Hans_nancount
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,1234,43.091163,23.492186,15.145734,33.428698,190.869670,...,0,0,0,0,0,0,0,0,0,0
1,Qwen/Qwen3-8B,gptq,2-bit,English,1234,36043.204264,34360.155089,34696.413729,13062.876244,42009.999499,...,897,948,935,965,719,993,984,939,882,998
135,Qwen/Qwen3-8B,gptq,2-bit,English,1000,NaN,NaN,NaN,12451.018302,NaN,...,0,0,0,0,0,0,674,860,918,988
136,Qwen/Qwen3-8B,gptq,2-bit,English,2000,NaN,NaN,NaN,12969.810741,NaN,...,0,0,0,0,0,0,925,923,943,996
6,Qwen/Qwen3-8B,gptq,3-bit,English,1234,81.769725,43.042818,21.217079,40.188512,368.785908,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230,meta-llama/Llama-3.1-8B-Instruct,sinq,2-bit,None,1234,NaN,NaN,NaN,8261.823109,NaN,...,0,0,0,0,0,0,0,0,0,0
229,meta-llama/Llama-3.1-8B-Instruct,sinq,3-bit,None,1234,NaN,NaN,NaN,50.553287,NaN,...,0,0,0,0,0,0,0,0,0,0
231,meta-llama/Llama-3.1-8B-Instruct,sinq,4-bit,None,1234,NaN,NaN,NaN,43.628489,NaN,...,0,0,0,0,0,0,0,0,0,0
233,Qwen/Qwen3-14B,sinq,2-bit,None,1234,NaN,NaN,NaN,93245.028692,NaN,...,0,0,0,0,0,0,0,0,0,0


Finish processing floresplus-germanic-results.csv with shape: (234, 41)


...

Finish processing GlobalMMLULite-results.csv with shape: (122, 9)


,Base Model,Quantization Technique,Bit Width,Calibration Language,Random Seed,dutch,german,tamil,indonesian,chinese
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,1234,0.680581,0.496403,0.403636,0.649091,0.739450
80,Qwen/Qwen3-8B,gptq,2-bit,English,1234,0.192377,0.280576,0.234545,0.240000,0.255046
76,Qwen/Qwen3-8B,gptq,3-bit,English,1234,0.401089,0.323741,0.245455,0.418182,0.526606
72,Qwen/Qwen3-8B,gptq,4-bit,English,1234,0.656987,0.496403,0.396364,0.649091,0.715596
82,Qwen/Qwen3-8B,gptq,2-bit,Tamil,1234,0.239564,0.208633,0.240000,0.274545,0.240367
...,...,...,...,...,...,...,...,...,...,...
115,meta-llama/Llama-3.1-8B-Instruct,sinq,2-bit,Tamil,1234,0.186933,0.266187,0.252727,0.272727,0.242202
111,meta-llama/Llama-3.1-8B-Instruct,sinq,3-bit,Tamil,1234,0.330309,0.453237,0.285455,0.409091,0.344954
108,meta-llama/Llama-3.1-8B-Instruct,sinq,4-bit,Tamil,1234,0.595281,0.474820,0.369091,0.569091,0.493578
118,Qwen/Qwen3-14B,sinq,2-bit,Tamil,1234,0.208711,0.302158,0.232727,0.227273,0.223853


Finish processing INCLUDE-results.csv with shape: (119, 10)


,Base Model,Quantization Technique,Bit Width,Calibration Language,Random Seed,en,ta,sw,zh
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,1234,0.848,0.716,0.392,0.880
33,Qwen/Qwen3-8B,gptq,2-bit,English,1234,0.012,0.008,0.004,0.012
13,Qwen/Qwen3-8B,gptq,3-bit,English,1234,0.508,0.196,0.104,0.492
9,Qwen/Qwen3-8B,gptq,4-bit,English,1234,0.864,0.512,0.348,0.852
34,Qwen/Qwen3-8B,gptq,2-bit,Tamil,1234,0.004,0.016,0.004,0.004
...,...,...,...,...,...,...,...,...,...
41,Qwen/Qwen3-14B,tacq,3-bit,Swahili,1234,0.908,0.664,0.576,0.852
40,Qwen/Qwen3-14B,tacq,2-bit,Chinese,1234,0.232,0.000,0.000,0.236
42,Qwen/Qwen3-14B,tacq,3-bit,Chinese,1234,0.928,0.712,0.448,0.880
99,Qwen/Qwen3-14B,sinq,2-bit,None,1234,0.000,0.012,0.008,0.012


Finish processing MGSM-results.csv with shape: (125, 9)


## Deliverance Line Function

### Bit

In [ ]:
import wandb
from wandb.apis.public import Api
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.subplots as ps
import plotly.graph_objects as go

def deliver_line(list_eval_languages, list_runs, list_base_models, list_eval_dataset, bits, custom_title="", display_mean=True, compile=False, show_variance=False, quantile=None, write_pdf=""):
  # Ensure that bits is a list not list of list
  if isinstance(bits, list):
    if len(bits) > 0 and isinstance(bits[0], list):
      bits = bits[max(enumerate([len(bit) for bit in bits]), key=lambda x: x[1])[0]]

  language_sort = {
      # Multi Blimp
      "eng": 0, "tam": 6, "idn": 8, "swa": 10, "zhoblimp": 12,
      # Belebele
      "eng_Latn": 0, "nld_Latn": 0.2, "deu_Latn": 0.4, "mal_Mlym": 5.8, "tam_Taml": 6, "tel_Telu": 7, "ind_Latn": 8, "tgl_Latn": 8.5, "zsm_Latn": 9, "swh_Latn": 10, "yor_Latn": 11, "lug_Latn": 10.5, "bod_Tibt": 12.2, "mya_Mymr": 12.4, "zho_Hans": 12,
      # Massive
      'en-US': 0, 'ta-IN': 6, 'id-ID': 8, 'sw-KE': 10, 'zh-CN': 12,
      # Global PIQA
      "eng_Latn": 0, "tam_Taml": 6, "ind_Latn": 8, "swh_Latn": 10, "cmn_Hans": 12,\
      # INCLUDE
      "french": 1, "hindi": 2, "nepali": 3, "albanian": 4, "tamil": 6, "telugu": 7, "indonesian": 8, "malay": 9, "chinese": 12, "japanese": 13, "korean": 14, "arabic": 15,\
      "dutch": 0.2, "german": 0.4, "tagalog": 8.5, "malayalam": 7.5,\
      # Global MMLU Lite
      "en": 0, "fr": 1, "hi": 2, "bn": 5, "id": 8, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,
  }

  custom_sort = {'Unquantized': 0, 'English': 1, 'Tamil': 2, 'Indonesian': 3, 'Swahili': 4, 'Chinese': 5,\
                 'Unquantized': 0, 'gptq': 1, 'slimllm': 2, 'tacq': 3,\
                 "Qwen/Qwen3-8B": 0, "meta-llama/Llama-3.1-8B-Instruct": 1, "CohereLabs/aya-expanse-8b": 2,\
                 "32-bit": 0, "2-bit": 1, "3-bit": 2, "4-bit": 3}

  calib_label_list = ["EN", "TA", "ID", "SW", "ZH"]
  calib_label = {"English": "EN", "Tamil": "TA", "Indonesian": "ID", "Swahili": "SW", "Chinese": "ZH"}
  calib_label_reversed = {v: k for k, v in calib_label.items()}
  base_model_label = {"Qwen/Qwen3-8B": "Qwen3", "meta-llama/Llama-3.1-8B-Instruct": "Llama", "CohereLabs/aya-expanse-8b": "Aya"}
  eval_label = {
      # Multi Blimp
      "eng": "EN", "tam": "TA", "idn": "ID", "swa": "SW", "zhoblimp": "ZH",
      # Belebele
      "eng_Latn": "EN", "nld_Latn": "NL", "deu_Latn": "DE", "mal_Mlym": "ML", "tam_Taml": "TA", "tel_Telu": "TE", "ind_Latn": "ID", "tgl_Latn": "TL", "zsm_Latn": "MS", "swh_Latn": "SW", "yor_Latn": "YO", "lug_Latn": "LG", "bod_Tibt": "BO", "mya_Mymr": "MY", "zho_Hans": "ZH",
      # Massive
      'en-US': "EN", 'id-ID': "ID", 'ta-IN': "TA", 'sw-KE': "SW", 'zh-CN': "ZH",
      # Global PIQA
      "eng_latn": "EN", "tam_taml": "TA", "ind_latn": "ID", "swh_latn": "SW", "cmn_hans": "ZH",\
      # INCLUDE
      "french": "FR", "hindi": "HI", "nepali": "NE", "albanian": "SQ", "tamil": "TA", "telugu": "TE", "indonesian": "ID", "malay": "MS", "chinese": "ZH", "japanese": "JA", "korean": "KO", "arabic": "AR",\
      "dutch": "NL", "german": "DE", "tagalog": "TL", "malayalam": "ML",\
      "english": "EN",\
      # Non-existance, but for the sake of labeling
      # Global MMLU Lite
      "en": "EN", "fr": "FR", "hi": "HI", "bn": "BN", "id": "ID", "sw": "SW", "yo": "YO", "zh": "ZH", "ja": "JA", "ko": "KO", "ar": "AR",
      # Multi Blimp
      "eng": "EN", "tam": "TA", "idn": "ID", "swa": "SW", "zhoblimp": "ZH",
  }
  family_mapper = {
      # INCLUDE
      "tamil": "TA", "telugu": "TA", "indonesian": "ID", "malay": "ID", "chinese": "ZH",\
      "dutch": "EN", "german": "EN", "tagalog": "ID", "malayalam": "TA",\
      "english": "EN",\
      # Non-existance, but for the sake of labeling
      # Global MMLU Lite
      "en": "EN", "de": "EN", "my": "ID", "id": "ID", "sw": "SW", "yo": "SW", "zh": "ZH",
  }

  # Figure for Line Plots Instances
  import plotly.graph_objects as go
  fig = go.Figure()

  # Chief loop of every tasks
  task_num = len(list_eval_dataset)

  for i in range(task_num):
    eval_languages = list_eval_languages[i]
    runs = list_runs[i]
    base_models = list_base_models[i]
    eval_dataset = list_eval_dataset[i]

    eval_languages = list(sorted(eval_languages, key=lambda x: language_sort.get(x, 100)))
    columns = ["Run Name", "Base Model", "Quantization Technique", "Bit Width", "Calibration Language", "Random Seed"]
    columns.extend(eval_languages)

    data_acc = []
    unquantized_data_acc = []

    # Iterate over the runs and print details
    for run in runs:
      config = run.config
      summary = run.summary

      if config["calibration_language"] == "WuuChinese":
        continue

      run_data_acc = []
      run_data_acc.extend([run.name, run.config["base_model"], config["quantization_technique"], config["bit_width"], config["calibration_language"], config.get("random_seed", 1234)])
      for lang in eval_languages:
        run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_acc_{lang}", None))

      if config["quantization_technique"] == "Unquantized":
        unquantized_data_acc.append(run_data_acc)
      else:
        data_acc.append(run_data_acc)

    df_unquantized_acc_chief = pd.DataFrame(unquantized_data_acc, columns=columns)
    df_acc_chief = pd.DataFrame(data_acc, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language', "Bit Width"], key=lambda x: x.map(custom_sort))

    display(df_unquantized_acc_chief)
    display(df_acc_chief)

    if eval_dataset.lower() == "include" and not compile and 'telugu' not in eval_languages:
      for df_acc in [df_acc_chief, df_unquantized_acc_chief]:
        df_acc.insert(6, 'EN', (df_acc['german'] + df_acc['dutch']) / (2))
        df_acc.drop(["german", "dutch"], axis=1, inplace=True)

      replace_index = min(eval_languages.index("german"), eval_languages.index("dutch"))
      eval_languages.insert(replace_index, "english")
      eval_languages.remove("german")
      eval_languages.remove("dutch")
    if eval_dataset.lower() == "globalpiqa":
      eval_dataset = eval_dataset + " v0.1"
    print(eval_dataset, eval_languages)


    random_seeds = df_acc_chief["Random Seed"].unique()
    # Calculate the direct - cross of aggregated bits, base_models, and quantization technique, after mean-ing languages
    # For loop for each: bits, base_models, and quantization technique
    aggregated_scores = []
    aggregated_variances = []
    aggregated_quantile = []
    for bit in bits:
      # Iterate for each model for decoupling
      df_chief = pd.DataFrame([])
      for random_seed in random_seeds:
        for base_model in base_models:
          df_acc = df_acc_chief.loc[df_acc_chief["Base Model"] == base_model].loc[df_acc_chief["Bit Width"] == bit][df_acc_chief["Random Seed"] == random_seed]
          df_unquantized_acc = df_unquantized_acc_chief.loc[df_unquantized_acc_chief["Base Model"] == base_model]
          base_model = base_model_label[base_model]

          df_acc.iloc[:, 6:] = df_acc.iloc[:, 6:]
          df_acc = df_acc.rename(lambda x: eval_label.get(x, x), axis='columns')
          df_acc.loc[:, "Calibration Language"] = df_acc.loc[:, "Calibration Language"].apply(lambda x: calib_label.get(x, x))
          df_acc[df_acc.columns[6:]] = df_acc[df_acc.columns[6:]].astype(float)

          # df_unquantized_acc.iloc[:, 5:] = df_unquantized_acc.iloc[:, 5:]
          df_unquantized_acc = df_unquantized_acc.rename(lambda x: eval_label.get(x, x), axis='columns')

          # The df structure: Column the eval, index (5 first columns) is the settings
          import numpy as np

          ## Placeholder for Direct, Cross
          df_acc["direct_transposed"] = np.empty(len(df_acc))
          df_acc["cross_transposed"] = np.empty(len(df_acc))

          ## Calculate Direct as En -> EN, while Cross as mean of ~En -> EN
          for quantization_technique in df_acc["Quantization Technique"].unique():
            df_filter = df_acc.loc[df_acc["Quantization Technique"] == quantization_technique]
            mean_ppl = df_filter.iloc[:, 6:6+len(eval_languages)].sum(axis=0)                                    # Seek the column using iloc, because seeking using loc is harder

            # drop if eval_language > calibration choices
            mean_ppl = mean_ppl.drop(mean_ppl.index[~mean_ppl.index.isin(df_filter["Calibration Language"])], axis=0)

            df_acc.loc[df_filter.index, "direct_transposed"] = [df_filter.iloc[i, 6 + [eval_label.get(lang, lang) for lang in eval_languages].index(calib)] for i, calib in enumerate(df_filter["Calibration Language"].unique())]
            df_acc.loc[df_filter.index, "cross_transposed"] = (mean_ppl.values - df_acc.loc[df_filter.index, "direct_transposed"].values) / (len(df_filter) - 1)
          # Finish loop for one models in a bit of a run
          df_chief = pd.concat([df_chief, df_acc], axis=0)

        # Finish loop for one bit, then add to respective
        score = (df_chief.loc[:, "direct_transposed"] - df_chief.loc[:, "cross_transposed"]).mean()
        aggregated_scores.append(score)
        if quantile is not None:
          aggregated_quantile.append((df_chief.loc[:, "direct_transposed"] - df_chief.loc[:, "cross_transposed"]).quantile(quantile))
        if show_variance:
          variance = (df_chief.loc[:, "direct_transposed"] - df_chief.loc[:, "cross_transposed"]).var()
          aggregated_variances.append(variance)

    # Append the bits's score to figure, then end for one eval/run
    color = px.colors.qualitative.Plotly[i]
    if quantile is None:
      fig.add_trace(go.Scatter(x=bits, y=aggregated_scores,
                      mode='lines+markers',
                      line_color=color,
                      showlegend=True,
                      name = eval_dataset,
                    ))
    elif quantile is not None:
      aggregated_scores = np.array(aggregated_scores)
      y_max = aggregated_quantile
      fig.add_trace(go.Scatter(
          x=bits,
          y=y_max,
          mode='lines+markers',
          line_color=color,
          showlegend=True,
          name = eval_dataset,
      ))

    if show_variance:
      print(aggregated_scores)
      print(aggregated_variances)
      aggregated_scores = np.array(aggregated_scores)
      aggregated_variances = np.array(aggregated_variances)
      y_upper = aggregated_scores + np.sqrt(aggregated_variances)
      y_lower = (aggregated_scores - np.sqrt(aggregated_variances))[::-1]
      fig.add_trace(go.Scatter(
          x=bits + bits[::-1],
          y=np.concatenate([y_upper, y_lower]),
          fill='toself',
          fillcolor=f'rgba({int(color[1:3], 16)},{int(color[3:5], 16)},{int(color[5:7], 16)},0.2)',
          line_color='rgba(255,255,255,0)',
          showlegend=False,
          name=eval_dataset,
      ))

    fig.update_layout(yaxis_title="Δ Direct- and<br> Cross-Calibrations")

    fig.update_xaxes(automargin=False)
    fig.update_yaxes(automargin=False)
    fig.update_layout(
        margin=dict(t=0, b=15 if not write_pdf else 25, l=70, r=0)
    )
    fig.layout.height = 60 * len(bits) + (15 if not write_pdf else 25)
    fig.layout.width = 70 + 120 * len(bits) + 0

    if quantile is not None:
      fig.update_yaxes(range=[-0.01, None])

    # # big_fig.update_layout(showlegend=False)

    fig.update_layout(legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99,
    ))

    # #
    # for i, base_model in enumerate(["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]):
    #   big_fig.update_yaxes(title_text=base_model_label[base_model], row=i+1, col=1)
  fig.show()
  if write_pdf:
    fig.write_image(write_pdf)

In [ ]:
# Initialize the W&B API
api = Api()

# Parameters
list_eval_languages = []
list_runs = []
list_base_models = []
list_eval_dataset = []
list_bits = []

# MultiBlimp
eval_languages = ["eng", "tam", "zhoblimp"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "MultiBlimp"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              # {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              # {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.calibration_language": "Indonesian"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Massive Scenario
eval_languages = ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN']
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
eval_dataset = "Massive"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                              # {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario-1000"},
                              # {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# XStoryCloze
eval_languages = ["en", "id", "zh"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
eval_dataset = "XStoryCloze"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              # {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              # {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# # m ARC
# eval_languages = ["en", "id", "zh"]
# num_samples = 128
# base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
# eval_dataset = "ARC"
# bits = ["2-bit", "3-bit", "4-bit"]

# runs = api.runs(path=f"{ENTITY}/{PROJECT}",
#                 filters={"$or": [
#                         { "$and": [
#                           {"config.num_calibration_samples": num_samples},
#                           {"config.evaluation_dataset": eval_dataset.lower()},
#                           {"config.enable_thinking": False},
#                           {"config.output_type": "multiple_choice"},
#                           {"$not": {"config.calibration_language": "WuuChinese"}},
#                           {"$not": {"config.calibration_language": "Swahili"}},
#                           {"$not": {"config.calibration_language": "Tamil"}},
#                           # {"$not": {"config.bit_width": "2-bit"}},
#                         ]
#                         },
#                         { "$and" : [
#                             {"config.quantization_technique": "Unquantized"},
#                           {"config.evaluation_dataset": eval_dataset.lower()},
#                             {"config.enable_thinking": False},
#                         ]}
#                     ]}
#                 )

# list_eval_languages.append(eval_languages)
# list_runs.append(runs)
# list_base_models.append(base_models)
# list_eval_dataset.append(eval_dataset)
# list_bits.append(bits)

# Belebele
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "Belebele"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              # {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              # {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Global MMLU Lite
eval_languages = ["en", "zh", "id", "sw"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "GlobalMMLULite"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              # {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              # {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B-Base"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B-Instruct-2507"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B-Base"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B-Instruct-2507"}},

                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# INCLUDE
eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Tamil"]
eval_languages = [eval_lang.lower() for eval_lang in eval_languages]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "INCLUDE"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              # {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              # {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Run the function
print("mean")
deliver_line(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, custom_title="", display_mean=False, compile=False, show_variance=False, quantile=None, write_pdf="summary_tasks.pdf")

print("mean + variance")
deliver_line(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, custom_title="", display_mean=False, compile=False, show_variance=True, quantile=None)

print("median")
deliver_line(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, custom_title="", display_mean=False, compile=False, show_variance=False, quantile=0.5)

print("q 75")
deliver_line(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, custom_title="", display_mean=False, compile=False, show_variance=False, quantile=0.75)

mean


,Run Name,Base Model,Quantization Technique,Bit Width,Calibration Language,Random Seed,eng,tam,zhoblimp
0,Qwen3-8B-Unquantized-32bit-Unquantized-multibl...,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,1234,0.972727,0.903141,0.819153
1,Llama-3.1-8B-Instruct-Unquantized-32bit-Unquan...,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,1234,0.976623,0.918848,0.795395
2,aya-expanse-8b-Unquantized-32bit-Unquantized-m...,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,1234,0.964935,0.921466,0.789576


,Run Name,Base Model,Quantization Technique,Bit Width,Calibration Language,Random Seed,eng,tam,zhoblimp
34,Qwen3-8B-gptq-2bit-English-multiblimp-nothink-...,Qwen/Qwen3-8B,gptq,2-bit,English,1234,0.681818,0.620419,0.562034
37,Qwen3-8B-gptq-3bit-English-multiblimp-nothink-...,Qwen/Qwen3-8B,gptq,3-bit,English,1234,0.977922,0.814136,0.804322
40,Qwen3-8B-gptq-4bit-English-multiblimp-nothink-...,Qwen/Qwen3-8B,gptq,4-bit,English,1234,0.997403,0.895288,1.000000
35,Qwen3-8B-gptq-2bit-Tamil-multiblimp-nothink-mu...,Qwen/Qwen3-8B,gptq,2-bit,Tamil,1234,0.624675,0.706806,0.567147
38,Qwen3-8B-gptq-3bit-Tamil-multiblimp-nothink-mu...,Qwen/Qwen3-8B,gptq,3-bit,Tamil,1234,0.972727,0.908377,0.821610
...,...,...,...,...,...,...,...,...,...
28,aya-expanse-8b-tacq-3bit-Tamil-multiblimp-noth...,CohereLabs/aya-expanse-8b,tacq,3-bit,Tamil,1234,0.944156,0.924084,0.768785
74,aya-expanse-8b-tacq-4bit-Tamil-multiblimp-noth...,CohereLabs/aya-expanse-8b,tacq,4-bit,Tamil,1234,0.958442,0.924084,0.785141
26,aya-expanse-8b-tacq-2bit-Chinese-multiblimp-no...,CohereLabs/aya-expanse-8b,tacq,2-bit,Chinese,1234,0.948052,0.748691,0.786384
29,aya-expanse-8b-tacq-3bit-Chinese-multiblimp-no...,CohereLabs/aya-expanse-8b,tacq,3-bit,Chinese,1234,0.953247,0.913613,0.786412


MultiBlimp ['eng', 'tam', 'zhoblimp']


/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Ser

...

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Ser

,Run Name,Base Model,Quantization Technique,Bit Width,Calibration Language,Random Seed,dutch,german,tamil,indonesian,chinese
0,Qwen3-8B-Unquantized-32bit-Unquantized-include...,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,1234,0.680581,0.496403,0.403636,0.649091,0.739450
1,aya-expanse-8b-Unquantized-32bit-Unquantized-i...,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,1234,0.631579,0.446043,0.283636,0.543636,0.500917
2,Llama-3.1-8B-Instruct-Unquantized-32bit-Unquan...,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,1234,0.627949,0.474820,0.390909,0.598182,0.537615


,Run Name,Base Model,Quantization Technique,Bit Width,Calibration Language,Random Seed,dutch,german,tamil,indonesian,chinese
77,Qwen3-8B-gptq-2bit-English-include-nothink-mul...,Qwen/Qwen3-8B,gptq,2-bit,English,1234,0.192377,0.280576,0.234545,0.240000,0.255046
73,Qwen3-8B-gptq-3bit-English-include-nothink-mul...,Qwen/Qwen3-8B,gptq,3-bit,English,1234,0.401089,0.323741,0.245455,0.418182,0.526606
69,Qwen3-8B-gptq-4bit-English-include-nothink-mul...,Qwen/Qwen3-8B,gptq,4-bit,English,1234,0.656987,0.496403,0.396364,0.649091,0.715596
79,Qwen3-8B-gptq-2bit-Tamil-include-nothink-multi...,Qwen/Qwen3-8B,gptq,2-bit,Tamil,1234,0.239564,0.208633,0.240000,0.274545,0.240367
75,Qwen3-8B-gptq-3bit-Tamil-include-nothink-multi...,Qwen/Qwen3-8B,gptq,3-bit,Tamil,1234,0.548094,0.424460,0.340000,0.509091,0.596330
...,...,...,...,...,...,...,...,...,...,...,...
58,aya-expanse-8b-tacq-3bit-Indonesian-include-no...,CohereLabs/aya-expanse-8b,tacq,3-bit,Indonesian,1234,0.578947,0.424460,0.278182,0.541818,0.478899
44,aya-expanse-8b-tacq-4bit-Indonesian-include-no...,CohereLabs/aya-expanse-8b,tacq,4-bit,Indonesian,1234,0.613430,0.395683,0.278182,0.552727,0.519266
40,aya-expanse-8b-tacq-2bit-Chinese-include-nothi...,CohereLabs/aya-expanse-8b,tacq,2-bit,Chinese,1234,0.326679,0.316547,0.280000,0.361818,0.339450
62,aya-expanse-8b-tacq-3bit-Chinese-include-nothi...,CohereLabs/aya-expanse-8b,tacq,3-bit,Chinese,1234,0.555354,0.438849,0.290909,0.536364,0.493578


INCLUDE ['english', 'tamil', 'indonesian', 'chinese']


/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_10019/761534824.py:139: UserWarning:

Boolean Ser

### Quantization

In [ ]:
import wandb
from wandb.apis.public import Api
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.subplots as ps
import plotly.graph_objects as go

def deliver_line_quantization_technique(list_eval_languages, list_runs, list_base_models, list_eval_dataset, bits, quantization_techniques, custom_title="", display_mean=True, compile=False, show_variance=False, quantile=None, write_pdf=""):
  # Ensure that bits is a list not list of list
  if isinstance(bits, list):
    if len(bits) > 0 and isinstance(bits[0], list):
      bits = bits[max(enumerate([len(bit) for bit in bits]), key=lambda x: x[1])[0]]

  if isinstance(quantization_techniques, list):
    if len(quantization_techniques) > 0 and isinstance(quantization_techniques[0], list):
      quantization_techniques = quantization_techniques[max(enumerate([len(qt) for qt in quantization_techniques]), key=lambda x: x[1])[0]]

  language_sort = {
      # Multi Blimp
      "eng": 0, "tam": 6, "idn": 8, "swa": 10, "zhoblimp": 12,
      # Belebele
      "eng_Latn": 0, "nld_Latn": 0.2, "deu_Latn": 0.4, "mal_Mlym": 5.8, "tam_Taml": 6, "tel_Telu": 7, "ind_Latn": 8, "tgl_Latn": 8.5, "zsm_Latn": 9, "swh_Latn": 10, "yor_Latn": 11, "lug_Latn": 10.5, "bod_Tibt": 12.2, "mya_Mymr": 12.4, "zho_Hans": 12,
      # Massive
      'en-US': 0, 'ta-IN': 6, 'id-ID': 8, 'sw-KE': 10, 'zh-CN': 12,
      # Global PIQA
      "eng_Latn": 0, "tam_Taml": 6, "ind_Latn": 8, "swh_Latn": 10, "cmn_Hans": 12,\
      # INCLUDE
      "french": 1, "hindi": 2, "nepali": 3, "albanian": 4, "tamil": 6, "telugu": 7, "indonesian": 8, "malay": 9, "chinese": 12, "japanese": 13, "korean": 14, "arabic": 15,\
      "dutch": 0.2, "german": 0.4, "tagalog": 8.5, "malayalam": 7.5,\
      # Global MMLU Lite
      "en": 0, "fr": 1, "hi": 2, "bn": 5, "id": 8, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,
  }

  custom_sort = {'Unquantized': 0, 'English': 1, 'Tamil': 2, 'Indonesian': 3, 'Swahili': 4, 'Chinese': 5,\
                 'Unquantized': 0, 'gptq': 1, 'slimllm': 2, 'tacq': 3,\
                 "Qwen/Qwen3-8B": 0, "meta-llama/Llama-3.1-8B-Instruct": 1, "CohereLabs/aya-expanse-8b": 2,\
                 "32-bit": 0, "2-bit": 1, "3-bit": 2, "4-bit": 3}

  calib_label_list = ["EN", "TA", "ID", "SW", "ZH"]
  calib_label = {"English": "EN", "Tamil": "TA", "Indonesian": "ID", "Swahili": "SW", "Chinese": "ZH"}
  calib_label_reversed = {v: k for k, v in calib_label.items()}
  base_model_label = {"Qwen/Qwen3-8B": "Qwen3", "meta-llama/Llama-3.1-8B-Instruct": "Llama", "CohereLabs/aya-expanse-8b": "Aya"}
  eval_label = {
      # Multi Blimp
      "eng": "EN", "tam": "TA", "idn": "ID", "swa": "SW", "zhoblimp": "ZH",
      # Belebele
      "eng_Latn": "EN", "nld_Latn": "NL", "deu_Latn": "DE", "mal_Mlym": "ML", "tam_Taml": "TA", "tel_Telu": "TE", "ind_Latn": "ID", "tgl_Latn": "TL", "zsm_Latn": "MS", "swh_Latn": "SW", "yor_Latn": "YO", "lug_Latn": "LG", "bod_Tibt": "BO", "mya_Mymr": "MY", "zho_Hans": "ZH",
      # Massive
      'en-US': "EN", 'id-ID': "ID", 'ta-IN': "TA", 'sw-KE': "SW", 'zh-CN': "ZH",
      # Global PIQA
      "eng_latn": "EN", "tam_taml": "TA", "ind_latn": "ID", "swh_latn": "SW", "cmn_hans": "ZH",\
      # INCLUDE
      "french": "FR", "hindi": "HI", "nepali": "NE", "albanian": "SQ", "tamil": "TA", "telugu": "TE", "indonesian": "ID", "malay": "MS", "chinese": "ZH", "japanese": "JA", "korean": "KO", "arabic": "AR",\
      "dutch": "NL", "german": "DE", "tagalog": "TL", "malayalam": "ML",\
      "english": "EN",\
      # Non-existance, but for the sake of labeling
      # Global MMLU Lite
      "en": "EN", "fr": "FR", "hi": "HI", "bn": "BN", "id": "ID", "sw": "SW", "yo": "YO", "zh": "ZH", "ja": "JA", "ko": "KO", "ar": "AR",
      # Multi Blimp
      "eng": "EN", "tam": "TA", "idn": "ID", "swa": "SW", "zhoblimp": "ZH",
  }
  family_mapper = {
      # INCLUDE
      "tamil": "TA", "telugu": "TA", "indonesian": "ID", "malay": "ID", "chinese": "ZH",\
      "dutch": "EN", "german": "EN", "tagalog": "ID", "malayalam": "TA",\
      "english": "EN",\
      # Non-existance, but for the sake of labeling
      # Global MMLU Lite
      "en": "EN", "de": "EN", "my": "ID", "id": "ID", "sw": "SW", "yo": "SW", "zh": "ZH",
  }

  # Figure for Line Plots Instances
  import plotly.graph_objects as go
  fig = go.Figure()

  # Chief loop of every tasks
  task_num = len(list_eval_dataset)

  for i in range(task_num):
    eval_languages = list_eval_languages[i]
    runs = list_runs[i]
    base_models = list_base_models[i]
    eval_dataset = list_eval_dataset[i]

    eval_languages = list(sorted(eval_languages, key=lambda x: language_sort.get(x, 100)))
    columns = ["Run Name", "Base Model", "Quantization Technique", "Bit Width", "Calibration Language"]
    columns.extend(eval_languages)

    data_acc = []
    unquantized_data_acc = []

    # Iterate over the runs and print details
    for run in runs:
      config = run.config
      summary = run.summary

      if config["calibration_language"] == "WuuChinese":
        continue

      run_data_acc = []
      run_data_acc.extend([run.name, run.config["base_model"], config["quantization_technique"], config["bit_width"], config["calibration_language"]])
      for lang in eval_languages:
        run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_acc_{lang}", None))

      if config["quantization_technique"] == "Unquantized":
        unquantized_data_acc.append(run_data_acc)
      else:
        data_acc.append(run_data_acc)

    df_unquantized_acc_chief = pd.DataFrame(unquantized_data_acc, columns=columns)
    df_acc_chief = pd.DataFrame(data_acc, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language', "Bit Width"], key=lambda x: x.map(custom_sort))

    if eval_dataset.lower() == "include" and not compile and 'telugu' not in eval_languages:
      for df_acc in [df_acc_chief, df_unquantized_acc_chief]:
        df_acc.insert(5, 'EN', (df_acc['german'] + df_acc['dutch']) / (2))
        df_acc.drop(["german", "dutch"], axis=1, inplace=True)

      replace_index = min(eval_languages.index("german"), eval_languages.index("dutch"))
      eval_languages.insert(replace_index, "english")
      eval_languages.remove("german")
      eval_languages.remove("dutch")
    if eval_dataset.lower() == "globalpiqa":
      eval_dataset = eval_dataset + " v0.1"
    print(eval_dataset, eval_languages)


    # Calculate the direct - cross of aggregated bits, base_models, and quantization technique, after mean-ing languages
    # For loop for each: bits, base_models, and quantization technique
    aggregated_scores = []
    aggregated_variances = []
    aggregated_quantile = []
    for quantization_technique in quantization_techniques:
      # Iterate for each model for decoupling
      df_chief = pd.DataFrame([])
      for base_model in base_models:
        df_acc = df_acc_chief.loc[df_acc_chief["Base Model"] == base_model].loc[df_acc_chief["Quantization Technique"] == quantization_technique]
        df_unquantized_acc = df_unquantized_acc_chief.loc[df_unquantized_acc_chief["Quantization Technique"] == quantization_technique]
        base_model = base_model_label[base_model]

        df_acc.iloc[:, 5:] = df_acc.iloc[:, 5:]
        df_acc = df_acc.rename(lambda x: eval_label.get(x, x), axis='columns')
        df_acc.loc[:, "Calibration Language"] = df_acc.loc[:, "Calibration Language"].apply(lambda x: calib_label.get(x, x))
        df_acc[df_acc.columns[5:]] = df_acc[df_acc.columns[5:]].astype(float)

        # df_unquantized_acc.iloc[:, 5:] = df_unquantized_acc.iloc[:, 5:]
        df_unquantized_acc = df_unquantized_acc.rename(lambda x: eval_label.get(x, x), axis='columns')

        # The df structure: Column the eval, index (5 first columns) is the settings
        import numpy as np

        ## Placeholder for Direct, Cross
        df_acc["direct_transposed"] = np.empty(len(df_acc))
        df_acc["cross_transposed"] = np.empty(len(df_acc))

        ## Calculate Direct as En -> EN, while Cross as mean of ~En -> EN
        for bit in bits:
          df_filter = df_acc.loc[df_acc["Bit Width"] == bit]
          mean_ppl = df_filter.iloc[:, 5:5+len(eval_languages)].sum(axis=0)                                    # Seek the column using iloc, because seeking using loc is harder

          # drop if eval_language > calibration choices
          mean_ppl = mean_ppl.drop(mean_ppl.index[~mean_ppl.index.isin(df_filter["Calibration Language"])], axis=0)

          df_acc.loc[df_filter.index, "direct_transposed"] = [df_filter.iloc[i, 5 + [eval_label.get(lang, lang) for lang in eval_languages].index(calib)] for i, calib in enumerate(df_filter["Calibration Language"].unique())]
          df_acc.loc[df_filter.index, "cross_transposed"] = (mean_ppl.values - df_acc.loc[df_filter.index, "direct_transposed"].values) / (len(df_filter) - 1)
        # Finish loop for one models in a bit of a run
        df_chief = pd.concat([df_chief, df_acc], axis=0)

      # Finish loop for one bit, then add to respective
      score = (df_chief.loc[:, "direct_transposed"] - df_chief.loc[:, "cross_transposed"]).mean()
      aggregated_scores.append(score)
      if quantile is not None:
        aggregated_quantile.append((df_chief.loc[:, "direct_transposed"] - df_chief.loc[:, "cross_transposed"]).quantile(quantile))
      if show_variance:
        variance = (df_chief.loc[:, "direct_transposed"] - df_chief.loc[:, "cross_transposed"]).var()
        aggregated_variances.append(variance)

    # Append the quantization techniques's score to figure, then end for one eval/run
    color = px.colors.qualitative.Plotly[i]
    if quantile is None:
      fig.add_trace(go.Scatter(x=quantization_techniques, y=aggregated_scores,
                      mode='lines+markers',
                      line_color=color,
                      showlegend=True,
                      name = eval_dataset,
                    ))
    elif quantile is not None:
      aggregated_scores = np.array(aggregated_scores)
      y_max = aggregated_quantile
      fig.add_trace(go.Scatter(
          x=quantization_techniques,
          y=y_max,
          mode='lines+markers',
          line_color=color,
          showlegend=True,
          name = eval_dataset,
      ))

    if show_variance:
      print(aggregated_scores)
      print(aggregated_variances)
      aggregated_scores = np.array(aggregated_scores)
      aggregated_variances = np.array(aggregated_variances)
      y_upper = aggregated_scores + np.sqrt(aggregated_variances)
      y_lower = (aggregated_scores - np.sqrt(aggregated_variances))[::-1]
      fig.add_trace(go.Scatter(
          x=quantization_techniques + quantization_techniques[::-1],
          y=np.concatenate([y_upper, y_lower]),
          fill='toself',
          fillcolor=f'rgba({int(color[1:3], 16)},{int(color[3:5], 16)},{int(color[5:7], 16)},0.2)',
          line_color='rgba(255,255,255,0)',
          showlegend=False,
          name=eval_dataset,
      ))

    fig.update_layout(yaxis_title="Δ Direct- and Cross-Calibrations")

    fig.update_xaxes(automargin=False)
    fig.update_yaxes(automargin=False)
    fig.update_layout(
        margin=dict(t=0, b=15 if not write_pdf else 25, l=50, r=0)
    )
    fig.layout.height = 75 * len(bits) + (15 if not write_pdf else 25)
    fig.layout.width = 50 + 150 * len(bits) + 0

    if quantile is not None:
      fig.update_yaxes(range=[-0.01, None])

    # # big_fig.update_layout(showlegend=False)

    fig.update_layout(legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99,
    ))

    # #
    # for i, base_model in enumerate(["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]):
    #   big_fig.update_yaxes(title_text=base_model_label[base_model], row=i+1, col=1)
  fig.show()
  if write_pdf:
    fig.write_image(write_pdf)

In [ ]:
# Initialize the W&B API
api = Api()

# Parameters
list_eval_languages = []
list_runs = []
list_base_models = []
list_eval_dataset = []
list_bits = []
list_quantization_techniques = []

# MultiBlimp
eval_languages = ["eng", "tam", "zhoblimp"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "MultiBlimp"
bits = ["2-bit", "3-bit", "4-bit"]
quantization_techniques = ["gptq", "slimllm", "tacq"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.calibration_language": "Indonesian"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_quantization_techniques.append(quantization_techniques)

# Massive Scenario
eval_languages = ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN']
num_samples = 128
base_models = ["Qwen/Qwen3-8B"]
eval_dataset = "Massive"
bits = ["2-bit", "3-bit"]
quantization_techniques = ["gptq", "slimllm", "tacq"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_quantization_techniques.append(quantization_techniques)

# Belebele
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "Belebele"
bits = ["2-bit", "3-bit", "4-bit"]
quantization_techniques = ["gptq", "slimllm", "tacq"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_quantization_techniques.append(quantization_techniques)

# m ARC
eval_languages = ["en", "id", "zh"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
eval_dataset = "ARC"
bits = ["3-bit", "4-bit"]
quantization_techniques = ["gptq", "slimllm", "tacq"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.bit_width": "2-bit"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_quantization_techniques.append(quantization_techniques)

# Global MMLU Lite
eval_languages = ["en", "zh", "id", "sw"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "GlobalMMLULite"
bits = ["2-bit", "3-bit", "4-bit"]
quantization_techniques = ["gptq", "slimllm", "tacq"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_quantization_techniques.append(quantization_techniques)

# INCLUDE
eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Tamil"]
eval_languages = [eval_lang.lower() for eval_lang in eval_languages]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "INCLUDE"
bits = ["2-bit", "3-bit", "4-bit"]
quantization_techniques = ["gptq", "slimllm", "tacq"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)
list_quantization_techniques.append(quantization_techniques)

# Run the function
print("mean")
deliver_line_quantization_technique(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, quantization_techniques=list_quantization_techniques, custom_title="", display_mean=False, compile=False, show_variance=False, quantile=None)

print("mean + variance")
deliver_line_quantization_technique(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, quantization_techniques=list_quantization_techniques, custom_title="", display_mean=False, compile=False, show_variance=True, quantile=None)

print("median")
deliver_line_quantization_technique(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, quantization_techniques=list_quantization_techniques, custom_title="", display_mean=False, compile=False, show_variance=False, quantile=0.5)

print("q 75")
deliver_line_quantization_technique(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, quantization_techniques=list_quantization_techniques, custom_title="", display_mean=False, compile=False, show_variance=False, quantile=0.75)

mean
MultiBlimp ['eng', 'tam', 'zhoblimp']
Massive ['en-US', 'ta-IN', 'id-ID', 'sw-KE', 'zh-CN']
Belebele ['eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn', 'zho_Hans']
ARC ['en', 'id', 'zh']
GlobalMMLULite ['en', 'id', 'sw', 'zh']
INCLUDE ['english', 'tamil', 'indonesian', 'chinese']


mean + variance
MultiBlimp ['eng', 'tam', 'zhoblimp']
[np.float64(0.05061906040275905), np.float64(0.08093959362619516), np.float64(0.06647167024973394)]
[0.0031147226915098484, 0.020317817189809417, 0.004865668745803485]
Massive ['en-US', 'ta-IN', 'id-ID', 'sw-KE', 'zh-CN']
[np.float64(0.015274041694687279), np.float64(0.022511768661735047), np.float64(0.058607935440484205)]
[0.0009995373519366651, 0.001396935474704405, 0.0035881453807164155]
Belebele ['eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn', 'zho_Hans']
[np.float64(0.025730452674897125), np.float64(0.025230078563411917), np.float64(0.030545267489711956)]
[0.0026683008905436898, 0.004129728288543874, 0.002842574847039867]
ARC ['en', 'id', 'zh']
[np.float64(0.012466550415652219), np.float64(0.009226266636446273), np.float64(0.005201044796853169)]
[0.0005383276214730522, 0.0002836592593861363, 0.0001303779305173539]
GlobalMMLULite ['en', 'id', 'sw', 'zh']
[np.float64(0.012476851851851845), np.float64(0.005324074074074078), np.flo

median
MultiBlimp ['eng', 'tam', 'zhoblimp']
Massive ['en-US', 'ta-IN', 'id-ID', 'sw-KE', 'zh-CN']
Belebele ['eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn', 'zho_Hans']
ARC ['en', 'id', 'zh']
GlobalMMLULite ['en', 'id', 'sw', 'zh']
INCLUDE ['english', 'tamil', 'indonesian', 'chinese']


q 75
MultiBlimp ['eng', 'tam', 'zhoblimp']
Massive ['en-US', 'ta-IN', 'id-ID', 'sw-KE', 'zh-CN']
Belebele ['eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn', 'zho_Hans']
ARC ['en', 'id', 'zh']
GlobalMMLULite ['en', 'id', 'sw', 'zh']
INCLUDE ['english', 'tamil', 'indonesian', 'chinese']


## Deliverance Spearman Function

### Spearman

In [ ]:
import wandb
from wandb.apis.public import Api
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.subplots as ps
import plotly.graph_objects as go

def deliver_spearman(list_eval_languages, list_runs, list_base_models, list_eval_dataset, bits, custom_title="", display_mean=True, compile=False, show_variance=False, quantile=None, write_pdf="", corr_method="spearman"):
  # Ensure that bits is a list not list of list
  if isinstance(bits, list):
    if len(bits) > 0 and isinstance(bits[0], list):
      bits = bits[max(enumerate([len(bit) for bit in bits]), key=lambda x: x[1])[0]]

  language_sort = {
      # Multi Blimp
      "eng": 0, "tam": 6, "idn": 8, "swa": 10, "zhoblimp": 12,
      # Belebele
      "eng_Latn": 0, "nld_Latn": 0.2, "deu_Latn": 0.4, "mal_Mlym": 5.8, "tam_Taml": 6, "tel_Telu": 7, "ind_Latn": 8, "tgl_Latn": 8.5, "zsm_Latn": 9, "swh_Latn": 10, "yor_Latn": 11, "lug_Latn": 10.5, "bod_Tibt": 12.2, "mya_Mymr": 12.4, "zho_Hans": 12,
      # Massive
      'en-US': 0, 'ta-IN': 6, 'id-ID': 8, 'sw-KE': 10, 'zh-CN': 12,
      # Global PIQA
      "eng_Latn": 0, "tam_Taml": 6, "ind_Latn": 8, "swh_Latn": 10, "cmn_Hans": 12,\
      # INCLUDE
      "french": 1, "hindi": 2, "nepali": 3, "albanian": 4, "tamil": 6, "telugu": 7, "indonesian": 8, "malay": 9, "chinese": 12, "japanese": 13, "korean": 14, "arabic": 15,\
      "dutch": 0.2, "german": 0.4, "tagalog": 8.5, "malayalam": 7.5,\
      # Global MMLU Lite
      "en": 0, "fr": 1, "hi": 2, "bn": 5, "id": 8, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,
  }

  custom_sort = {'Unquantized': 0, 'English': 1, 'Tamil': 2, 'Indonesian': 3, 'Swahili': 4, 'Chinese': 5,\
                 'Unquantized': 0, 'gptq': 1, 'slimllm': 2, 'tacq': 3,\
                 "Qwen/Qwen3-8B": 0, "meta-llama/Llama-3.1-8B-Instruct": 1, "CohereLabs/aya-expanse-8b": 2,\
                 "32-bit": 0, "2-bit": 1, "3-bit": 2, "4-bit": 3}

  calib_label_list = ["EN", "TA", "ID", "SW", "ZH"]
  calib_label = {"English": "EN", "Tamil": "TA", "Indonesian": "ID", "Swahili": "SW", "Chinese": "ZH"}
  calib_label_reversed = {v: k for k, v in calib_label.items()}
  base_model_label = {"Qwen/Qwen3-8B": "Qwen3", "meta-llama/Llama-3.1-8B-Instruct": "Llama", "CohereLabs/aya-expanse-8b": "Aya"}
  eval_label = {
      # Multi Blimp
      "eng": "EN", "tam": "TA", "idn": "ID", "swa": "SW", "zhoblimp": "ZH",
      # Belebele
      "eng_Latn": "EN", "nld_Latn": "NL", "deu_Latn": "DE", "mal_Mlym": "ML", "tam_Taml": "TA", "tel_Telu": "TE", "ind_Latn": "ID", "tgl_Latn": "TL", "zsm_Latn": "MS", "swh_Latn": "SW", "yor_Latn": "YO", "lug_Latn": "LG", "bod_Tibt": "BO", "mya_Mymr": "MY", "zho_Hans": "ZH",
      # Massive
      'en-US': "EN", 'id-ID': "ID", 'ta-IN': "TA", 'sw-KE': "SW", 'zh-CN': "ZH",
      # Global PIQA
      "eng_latn": "EN", "tam_taml": "TA", "ind_latn": "ID", "swh_latn": "SW", "cmn_hans": "ZH",\
      # INCLUDE
      "french": "FR", "hindi": "HI", "nepali": "NE", "albanian": "SQ", "tamil": "TA", "telugu": "TE", "indonesian": "ID", "malay": "MS", "chinese": "ZH", "japanese": "JA", "korean": "KO", "arabic": "AR",\
      "dutch": "NL", "german": "DE", "tagalog": "TL", "malayalam": "ML",\
      "english": "EN",\
      # Non-existance, but for the sake of labeling
      # Global MMLU Lite
      "en": "EN", "fr": "FR", "hi": "HI", "bn": "BN", "id": "ID", "sw": "SW", "yo": "YO", "zh": "ZH", "ja": "JA", "ko": "KO", "ar": "AR",
      # Multi Blimp
      "eng": "EN", "tam": "TA", "idn": "ID", "swa": "SW", "zhoblimp": "ZH",
  }
  reverse_eval_label = {
      'multiblimp': {
          "English": "eng", "Tamil": "tam", "Indonesian": "idn", "Swahili": "swa", "Chinese": "zhoblimp",
      },
      'massive': {
          "English": 'en-US', "Indonesian": 'id-ID', "Tamil": 'ta-IN', "Swahili": 'sw-KE', "Chinese": 'zh-CN',
      },
      'xstorycloze': {
          "English": "en", "Indonesian": "id", "Swahili": "sw", "Chinese": "zh",
      },
      'belebele': {
          "English": "eng_Latn", "Tamil": "tam_Taml", "Indonesian": "ind_Latn", "Swahili": "swh_Latn", "Chinese": "zho_Hans",
      },
      'include': {
          "English": "english", "Chinese": "chinese", "Indonesian": "indonesian", "Tamil": "tamil",
      },
      'globalmmlulite': {
          "English": "en", "Chinese": "zh", "Indonesian": "id", "Swahili": "sw",
      },
  }
  family_mapper = {
      # INCLUDE
      "tamil": "TA", "telugu": "TA", "indonesian": "ID", "malay": "ID", "chinese": "ZH",\
      "dutch": "EN", "german": "EN", "tagalog": "ID", "malayalam": "TA",\
      "english": "EN",\
      # Non-existance, but for the sake of labeling
      # Global MMLU Lite
      "en": "EN", "de": "EN", "my": "ID", "id": "ID", "sw": "SW", "yo": "SW", "zh": "ZH",
  }
  # Mapper from categorical to ordinal
  base_model_mapper = {
      "Qwen/Qwen3-8B": 3,
      "CohereLabs/aya-expanse-8b": 2,
      "meta-llama/Llama-3.1-8B-Instruct": 1,
  }
  quantization_technique_mapper = {
      "gptq": 1,
      "slimllm": 2,
      "tacq": 3,
  }
  bit_width_mapper = {
      "4-bit": 3,
      "3-bit": 2,
      "2-bit": 1,
  }
  ## According to aya dataset, for now
  calibration_language_mapper = {
      "English": 3,
      "Tamil": 2,
      "Indonesian": 2,
      "Swahili": 1,
      "Chinese": 3,
  }
  task_difficulty_mapper = {
      "multiblimp": 1,
      "massive": 1,
      "xstorycloze": 1,
      "belebele": 2,
      "include": 2,
      "globalmmlulite": 2,
  }
  spearman_mapper = dict()
  spearman_mapper.update(base_model_mapper)
  spearman_mapper.update(quantization_technique_mapper)
  spearman_mapper.update(bit_width_mapper)
  spearman_mapper.update(calibration_language_mapper)
  spearman_mapper.update(task_difficulty_mapper)

  # Figure for Line Plots Instances
  import plotly.graph_objects as go
  fig = go.Figure()


  columns = ["Base Model", "Quantization Technique", "Bit Width", "Calibration Language", "Task Difficulty", "Direct", "Cross", "Delta"]
  df_chief = pd.DataFrame([], columns=columns)

  # Chief loop of every tasks
  task_num = len(list_eval_dataset)

  for i in range(task_num):
    eval_languages = list_eval_languages[i]
    runs = list_runs[i]
    base_models = list_base_models[i]
    eval_dataset = list_eval_dataset[i]
    print(f"Processing ... {eval_dataset}")

    eval_languages = list(sorted(eval_languages, key=lambda x: language_sort.get(x, 100)))
    columns = ["Base Model", "Quantization Technique", "Bit Width", "Calibration Language", "Task Difficulty"]
    columns.extend(eval_languages)

    data_acc = []

    # Iterate over the runs and print details
    for run in runs:
      config = run.config
      summary = run.summary

      if config["calibration_language"] == "WuuChinese":
        continue
      if config["quantization_technique"] == "Unquantized":
        continue

      run_data_acc = []
      run_data_acc.extend([run.config["base_model"], config["quantization_technique"], config["bit_width"], config["calibration_language"], eval_dataset.lower()])
      for lang in eval_languages:
        run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_acc_{lang}", None))

      data_acc.append(run_data_acc)

    df_acc_chief = pd.DataFrame(data_acc, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language', "Bit Width"], key=lambda x: x.map(custom_sort))

    # Agg Eng Eval on include
    if eval_dataset.lower() == "include" and not compile and 'telugu' not in eval_languages:
      for df_acc in [df_acc_chief]:
        df_acc.insert(5, 'english', (df_acc['german'] + df_acc['dutch']) / (2))
        df_acc.drop(["german", "dutch"], axis=1, inplace=True)
        replace_index = min(eval_languages.index("german"), eval_languages.index("dutch"))
        eval_languages.insert(replace_index, "english")
        eval_languages.remove("german")
        eval_languages.remove("dutch")

    # Calculate Direct, Cross, and Stats
    ## Groupby's apply is applied to each group's dataframe
    df_acc_chief_groupby = df_acc_chief.groupby(['Base Model', 'Quantization Technique', "Bit Width", "Task Difficulty"], as_index=False, group_keys=True)
    # df_acc_chief_groupby.apply(print)
    # Calib Language Series = x["Calibration Language"].apply(reverse_eval_label[eval_dataset.lower()]) -> Transform each Calib into eval dataset lang's name
    # display(df_acc_chief_groupby.apply(lambda x: x.loc[x["Calibration Language"], x["Calibration Language"].apply(reverse_eval_label[eval_dataset.lower()])]))
    # df_acc_chief_groupby.apply(lambda x: print(x["Calibration Language"].map(reverse_eval_label[eval_dataset.lower()]).to_list()))
    # df_acc_chief_groupby.apply(lambda x: print(x.index.to_list()))
    # df_acc_chief_stats = pd.DataFrame([], columns=["Direct", "Cross", "Count"])
    # df_acc_chief_groupby.apply(lambda x: print([x.loc[i, j] for i, j in zip(x.index.to_list(), x["Calibration Language"].map(reverse_eval_label[eval_dataset.lower()]).to_list())]))
    # display(df_acc_chief_groupby.apply(lambda x: x.loc[:, x["Calibration Language"].map(reverse_eval_label[eval_dataset.lower()])].sum(axis=0)))

    df_acc_chief["Direct"] = df_acc_chief_groupby.apply(lambda x: pd.Series([x.loc[i, j] for i, j in zip(x.index.to_list(), x["Calibration Language"].map(reverse_eval_label[eval_dataset.lower()]).to_list())], index=x.index)).reset_index(level=0, drop=True)
    df_acc_chief["Cross"] = df_acc_chief_groupby.apply(lambda x: x.loc[:, x["Calibration Language"].map(reverse_eval_label[eval_dataset.lower()])].sum(axis=0).set_axis(x.index)).reset_index(level=0, drop=True)
    df_acc_chief["Count"] = df_acc_chief_groupby.apply(lambda x: pd.Series(x["Calibration Language"].size, index=x.index)).reset_index(level=0, drop=True)
    #   {
    #     "Direct": df_acc_chief_groupby.apply(lambda x: [x.loc[i, j] for i, j in zip(x.index.to_list(), x["Calibration Language"].map(reverse_eval_label[eval_dataset.lower()]).to_list())]),
    #     "Cross": df_acc_chief_groupby.apply(lambda x: x.loc[x["Calibration Language"].apply(reverse_eval_label[eval_dataset.lower()])].sum()),
    #     "Count": df_acc_chief_groupby.apply(lambda x: x["Calibration Language"].count()),
    # })

    df_acc_chief["Cross"] = (df_acc_chief["Cross"] - df_acc_chief["Direct"]) / (df_acc_chief["Count"] - 1)
    df_acc_chief["Delta"] = df_acc_chief["Direct"] - df_acc_chief["Cross"]

    # Drop the unused columns later in spearman
    df_acc_chief = df_acc_chief.drop("Count", axis=1)
    df_acc_chief = df_acc_chief.drop(eval_languages, axis=1)

    # Convert all columns into numeric even if ordinal
    df_acc_chief = df_acc_chief.map(lambda x: spearman_mapper.get(x, x))

    # Concat back to the chief
    df_chief = pd.concat([df_chief, df_acc_chief], axis=0)

  display(df_chief)
  display(df_chief.corr(method=corr_method).style.background_gradient(cmap='coolwarm').format(precision=2))

  def display_df_cor(df, filter, value):
    display(df[df[filter] == value].corr(method=corr_method).drop(filter, axis=0).drop(filter, axis=1).style.background_gradient(cmap='coolwarm').format(precision=2))

  print("2-bit only")
  display_df_cor(df_chief, "Bit Width", 1)
  print("3-bit only")
  display_df_cor(df_chief, "Bit Width", 2)
  print("4-bit only")
  display_df_cor(df_chief, "Bit Width", 3)

  print("GPTQ only")
  display_df_cor(df_chief, "Quantization Technique", 1)
  print("SliMLLM only")
  display_df_cor(df_chief, "Quantization Technique", 2)
  print("TaCQ only")
  display_df_cor(df_chief, "Quantization Technique", 3)

  print("Low-Resource only")
  display_df_cor(df_chief, "Calibration Language", 1)
  print("Mid-Resource only")
  display_df_cor(df_chief, "Calibration Language", 2)
  print("High-Resource only")
  display_df_cor(df_chief, "Calibration Language", 3)

  print("Easy task only")
  display_df_cor(df_chief, "Task Difficulty", 1)
  print("Hard task only")
  display_df_cor(df_chief, "Task Difficulty", 2)

  # # Manipulated Columns
  # df_chief["Quant x Bit"] = df_chief["Bit Width"] * 3 + df_chief["Quantization Technique"]
  # display(df_chief.corr(method='spearman').style.background_gradient(cmap='coolwarm').format(precision=2))


  # df_chief["Quant x Bit x Task"] = df_chief["Quant x Bit"] * df_chief["Task Difficulty"]
  # display(df_chief.corr(method='spearman').style.background_gradient(cmap='coolwarm').format(precision=2))


  # # Pair-wise scatter plot
  # import matplotlib.pyplot as plt
  # # grr = pd.plotting.scatter_matrix(df_chief, figsize=(15, 15), marker='o',
  # #                                hist_kwds={'bins': 20}, s=60, alpha=.8)

  # # plt.show()

  # import seaborn as sns
  # sns.pairplot(df_chief)

  # plt.show()

  # # Manipulated Columns
  # df_chief["Quant x Bit"] = df_chief["Bit Width"] * 3 + df_chief["Quantization Technique"]
  # df_chief["Calib x Task"] = df_chief["Calibration Language"] + 5 * df_chief["Task Difficulty"]
  # sns.pairplot(df_chief)

  # plt.show()

  # df_chief["Quant x Bit x Calib x Task"] = df_chief["Quant x Bit"] * df_chief["Calib x Task"]
  # sns.pairplot(df_chief)

  # plt.show()

  # if write_pdf:
  #   fig.write_image(write_pdf)

In [ ]:
# Initialize the W&B API
api = Api()

# Parameters
list_eval_languages = []
list_runs = []
list_base_models = []
list_eval_dataset = []
list_bits = []

# MultiBlimp
eval_languages = ["eng", "tam", "zhoblimp"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "MultiBlimp"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.calibration_language": "Indonesian"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Massive Scenario
eval_languages = ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN']
num_samples = 128
base_models = ["Qwen/Qwen3-8B"]
eval_dataset = "Massive"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# XStoryCloze
eval_languages = ["en", "id", "zh"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
eval_dataset = "XStoryCloze"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# # Global PIQA v0.1
# eval_languages = ["eng_latn", "ind_latn", "cmn_hans"]
# num_samples = 128
# base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
# eval_dataset = "GlobalPIQA"
# bits = ["2-bit", "3-bit", "4-bit"]

# runs = api.runs(path=f"{ENTITY}/{PROJECT}",
#                 filters={"$or": [
#                         { "$and": [
#                           {"config.num_calibration_samples": num_samples},
#                           {"config.evaluation_dataset": eval_dataset.lower()},
#                           {"config.enable_thinking": False},
#                           {"config.output_type": "multiple_choice"},
#                           {"$not": {"config.calibration_language": "WuuChinese"}},
#                           {"$not": {"config.calibration_language": "Tamil"}},
#                           {"$not": {"config.calibration_language": "Swahili"}},
#                           # {"$not": {"config.bit_width": "2-bit"}},
#                           {"$not": {"config.bit_width": "8-bit"}},
#                         ]
#                         },
#                         { "$and" : [
#                             {"config.quantization_technique": "Unquantized"},
#                           {"config.evaluation_dataset": eval_dataset.lower()},
#                             {"config.enable_thinking": False},
#                         ]}
#                     ]}
#                 )

# list_eval_languages.append(eval_languages)
# list_runs.append(runs)
# list_base_models.append(base_models)
# list_eval_dataset.append(eval_dataset)
# list_bits.append(bits)

# # m ARC
# eval_languages = ["en", "id", "zh"]
# num_samples = 128
# base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
# eval_dataset = "ARC"
# bits = ["2-bit", "3-bit", "4-bit"]

# runs = api.runs(path=f"{ENTITY}/{PROJECT}",
#                 filters={"$or": [
#                         { "$and": [
#                           {"config.num_calibration_samples": num_samples},
#                           {"config.evaluation_dataset": eval_dataset.lower()},
#                           {"config.enable_thinking": False},
#                           {"config.output_type": "multiple_choice"},
#                           {"$not": {"config.calibration_language": "WuuChinese"}},
#                           {"$not": {"config.calibration_language": "Swahili"}},
#                           {"$not": {"config.calibration_language": "Tamil"}},
#                           # {"$not": {"config.bit_width": "2-bit"}},
#                         ]
#                         },
#                         { "$and" : [
#                             {"config.quantization_technique": "Unquantized"},
#                           {"config.evaluation_dataset": eval_dataset.lower()},
#                             {"config.enable_thinking": False},
#                         ]}
#                     ]}
#                 )

# list_eval_languages.append(eval_languages)
# list_runs.append(runs)
# list_base_models.append(base_models)
# list_eval_dataset.append(eval_dataset)
# list_bits.append(bits)

# Belebele
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "Belebele"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Global MMLU Lite
eval_languages = ["en", "zh", "id", "sw"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "GlobalMMLULite"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# INCLUDE
eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Tamil"]
eval_languages = [eval_lang.lower() for eval_lang in eval_languages]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "INCLUDE"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

print(f"{list_eval_languages=}")
print(f"{list_runs=}")
print(f"{list_base_models=}")
print(f"{list_eval_dataset=}")
print(f"{list_bits=}")
print()

# Run the function
deliver_spearman(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, custom_title="", display_mean=False, compile=False)

#### Easy task only

In [ ]:
# Initialize the W&B API
api = Api()

# Parameters
list_eval_languages = []
list_runs = []
list_base_models = []
list_eval_dataset = []
list_bits = []

# MultiBlimp
eval_languages = ["eng", "tam", "zhoblimp"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "MultiBlimp"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.calibration_language": "Indonesian"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Massive Scenario
eval_languages = ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN']
num_samples = 128
base_models = ["Qwen/Qwen3-8B"]
eval_dataset = "Massive"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# XStoryCloze
eval_languages = ["en", "id", "zh"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
eval_dataset = "XStoryCloze"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

print(f"{list_eval_languages=}")
print(f"{list_runs=}")
print(f"{list_base_models=}")
print(f"{list_eval_dataset=}")
print(f"{list_bits=}")
print()

# Run the function
deliver_spearman(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, custom_title="", display_mean=False, compile=False)

#### Hard task only

In [ ]:
# Initialize the W&B API
api = Api()

# Parameters
list_eval_languages = []
list_runs = []
list_base_models = []
list_eval_dataset = []
list_bits = []

# Belebele
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "Belebele"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Global MMLU Lite
eval_languages = ["en", "zh", "id", "sw"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "GlobalMMLULite"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# INCLUDE
eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Tamil"]
eval_languages = [eval_lang.lower() for eval_lang in eval_languages]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "INCLUDE"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

print(f"{list_eval_languages=}")
print(f"{list_runs=}")
print(f"{list_base_models=}")
print(f"{list_eval_dataset=}")
print(f"{list_bits=}")
print()

# Run the function
deliver_spearman(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, custom_title="", display_mean=False, compile=False)

#### No 4-bit

In [ ]:
# Initialize the W&B API
api = Api()

# Parameters
list_eval_languages = []
list_runs = []
list_base_models = []
list_eval_dataset = []
list_bits = []

# MultiBlimp
eval_languages = ["eng", "tam", "zhoblimp"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "MultiBlimp"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.calibration_language": "Indonesian"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Massive Scenario
eval_languages = ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN']
num_samples = 128
base_models = ["Qwen/Qwen3-8B"]
eval_dataset = "Massive"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# XStoryCloze
eval_languages = ["en", "id", "zh"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
eval_dataset = "XStoryCloze"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Belebele
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "Belebele"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Global MMLU Lite
eval_languages = ["en", "zh", "id", "sw"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "GlobalMMLULite"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# INCLUDE
eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Tamil"]
eval_languages = [eval_lang.lower() for eval_lang in eval_languages]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "INCLUDE"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

print(f"{list_eval_languages=}")
print(f"{list_runs=}")
print(f"{list_base_models=}")
print(f"{list_eval_dataset=}")
print(f"{list_bits=}")
print()

# Run the function
deliver_spearman(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, custom_title="", display_mean=False, compile=False, corr_method='spearman')

In [ ]:
# Initialize the W&B API
api = Api()

# Parameters
list_eval_languages = []
list_runs = []
list_base_models = []
list_eval_dataset = []
list_bits = []

# MultiBlimp
eval_languages = ["eng", "tam", "zhoblimp"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "MultiBlimp"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.calibration_language": "Indonesian"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Massive Scenario
eval_languages = ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN']
num_samples = 128
base_models = ["Qwen/Qwen3-8B"]
eval_dataset = "Massive"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# XStoryCloze
eval_languages = ["en", "id", "zh"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
eval_dataset = "XStoryCloze"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Belebele
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "Belebele"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Global MMLU Lite
eval_languages = ["en", "zh", "id", "sw"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "GlobalMMLULite"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# INCLUDE
eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Tamil"]
eval_languages = [eval_lang.lower() for eval_lang in eval_languages]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "INCLUDE"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

print(f"{list_eval_languages=}")
print(f"{list_runs=}")
print(f"{list_base_models=}")
print(f"{list_eval_dataset=}")
print(f"{list_bits=}")
print()

# Run the function
deliver_spearman(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, custom_title="", display_mean=False, compile=False, corr_method="kendall")

#### Lang as 5 ordinal

In [ ]:
import wandb
from wandb.apis.public import Api
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.subplots as ps
import plotly.graph_objects as go

def deliver_spearman_5langdegree(list_eval_languages, list_runs, list_base_models, list_eval_dataset, bits, custom_title="", display_mean=True, compile=False, show_variance=False, quantile=None, write_pdf="", corr_method="spearman", display_raw_score=True, truncate_heatmap=False, include_4bit=False, exclude_model=False, higher_task_difficulty=False):
  # Ensure that bits is a list not list of list
  if isinstance(bits, list):
    if len(bits) > 0 and isinstance(bits[0], list):
      bits = bits[max(enumerate([len(bit) for bit in bits]), key=lambda x: x[1])[0]]

  language_sort = {
      # Multi Blimp
      "eng": 0, "tam": 6, "idn": 8, "swa": 10, "zhoblimp": 12,
      # Belebele
      "eng_Latn": 0, "nld_Latn": 0.2, "deu_Latn": 0.4, "mal_Mlym": 5.8, "tam_Taml": 6, "tel_Telu": 7, "ind_Latn": 8, "tgl_Latn": 8.5, "zsm_Latn": 9, "swh_Latn": 10, "yor_Latn": 11, "lug_Latn": 10.5, "bod_Tibt": 12.2, "mya_Mymr": 12.4, "zho_Hans": 12,
      # Massive
      'en-US': 0, 'ta-IN': 6, 'id-ID': 8, 'sw-KE': 10, 'zh-CN': 12,
      # Global PIQA
      "eng_Latn": 0, "tam_Taml": 6, "ind_Latn": 8, "swh_Latn": 10, "cmn_Hans": 12,\
      # INCLUDE
      "french": 1, "hindi": 2, "nepali": 3, "albanian": 4, "tamil": 6, "telugu": 7, "indonesian": 8, "malay": 9, "chinese": 12, "japanese": 13, "korean": 14, "arabic": 15,\
      "dutch": 0.2, "german": 0.4, "tagalog": 8.5, "malayalam": 7.5,\
      # Global MMLU Lite
      "en": 0, "fr": 1, "hi": 2, "bn": 5, "id": 8, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,
  }

  custom_sort = {'Unquantized': 0, 'English': 1, 'Tamil': 2, 'Indonesian': 3, 'Swahili': 4, 'Chinese': 5,\
                 'Unquantized': 0, 'gptq': 1, 'slimllm': 2, 'tacq': 3,\
                 "Qwen/Qwen3-8B": 0, "meta-llama/Llama-3.1-8B-Instruct": 1, "CohereLabs/aya-expanse-8b": 2, "Qwen/Qwen3-8B": 3, \
                 "32-bit": 0, "2-bit": 1, "3-bit": 2, "4-bit": 3}

  calib_label_list = ["EN", "TA", "ID", "SW", "ZH"]
  calib_label = {"English": "EN", "Tamil": "TA", "Indonesian": "ID", "Swahili": "SW", "Chinese": "ZH"}
  calib_label_reversed = {v: k for k, v in calib_label.items()}
  base_model_label = {"Qwen/Qwen3-8B": "Qwen3", "meta-llama/Llama-3.1-8B-Instruct": "Llama", "CohereLabs/aya-expanse-8b": "Aya"}
  eval_label = {
      # Multi Blimp
      "eng": "EN", "tam": "TA", "idn": "ID", "swa": "SW", "zhoblimp": "ZH",
      # Belebele
      "eng_Latn": "EN", "nld_Latn": "NL", "deu_Latn": "DE", "mal_Mlym": "ML", "tam_Taml": "TA", "tel_Telu": "TE", "ind_Latn": "ID", "tgl_Latn": "TL", "zsm_Latn": "MS", "swh_Latn": "SW", "yor_Latn": "YO", "lug_Latn": "LG", "bod_Tibt": "BO", "mya_Mymr": "MY", "zho_Hans": "ZH",
      # Massive
      'en-US': "EN", 'id-ID': "ID", 'ta-IN': "TA", 'sw-KE': "SW", 'zh-CN': "ZH",
      # Global PIQA
      "eng_latn": "EN", "tam_taml": "TA", "ind_latn": "ID", "swh_latn": "SW", "cmn_hans": "ZH",\
      # INCLUDE
      "french": "FR", "hindi": "HI", "nepali": "NE", "albanian": "SQ", "tamil": "TA", "telugu": "TE", "indonesian": "ID", "malay": "MS", "chinese": "ZH", "japanese": "JA", "korean": "KO", "arabic": "AR",\
      "dutch": "NL", "german": "DE", "tagalog": "TL", "malayalam": "ML",\
      "english": "EN",\
      # Non-existance, but for the sake of labeling
      # Global MMLU Lite
      "en": "EN", "fr": "FR", "hi": "HI", "bn": "BN", "id": "ID", "sw": "SW", "yo": "YO", "zh": "ZH", "ja": "JA", "ko": "KO", "ar": "AR",
      # Multi Blimp
      "eng": "EN", "tam": "TA", "idn": "ID", "swa": "SW", "zhoblimp": "ZH",
  }
  reverse_eval_label = {
      'multiblimp': {
          "English": "eng", "Tamil": "tam", "Indonesian": "idn", "Swahili": "swa", "Chinese": "zhoblimp",
      },
      'massive': {
          "English": 'en-US', "Indonesian": 'id-ID', "Tamil": 'ta-IN', "Swahili": 'sw-KE', "Chinese": 'zh-CN',
      },
      'xstorycloze': {
          "English": "en", "Indonesian": "id", "Swahili": "sw", "Chinese": "zh",
      },
      'belebele': {
          "English": "eng_Latn", "Tamil": "tam_Taml", "Indonesian": "ind_Latn", "Swahili": "swh_Latn", "Chinese": "zho_Hans",
      },
      'include': {
          "English": "english", "Chinese": "chinese", "Indonesian": "indonesian", "Tamil": "tamil",
      },
      'globalmmlulite': {
          "English": "en", "Chinese": "zh", "Indonesian": "id", "Swahili": "sw",
      },
  }
  family_mapper = {
      # INCLUDE
      "tamil": "TA", "telugu": "TA", "indonesian": "ID", "malay": "ID", "chinese": "ZH",\
      "dutch": "EN", "german": "EN", "tagalog": "ID", "malayalam": "TA",\
      "english": "EN",\
      # Non-existance, but for the sake of labeling
      # Global MMLU Lite
      "en": "EN", "de": "EN", "my": "ID", "id": "ID", "sw": "SW", "yo": "SW", "zh": "ZH",
  }
  # Mapper from categorical to ordinal
  base_model_mapper = {
      "Qwen/Qwen3-8B": 3,
      "CohereLabs/aya-expanse-8b": 2,
      "meta-llama/Llama-3.1-8B-Instruct": 1,
  }
  quantization_technique_mapper = {
      # "sinq": 1,
      "gptq": 1,
      "slimllm": 2,
      "tacq": 3,
  }
  bit_width_mapper = {
      "4-bit": 3,
      "3-bit": 2,
      "2-bit": 1,
  }
  ## According to aya dataset, for now
  calibration_language_mapper = {
      "English": 5,
      "Tamil": 2,
      "Indonesian": 3,
      "Swahili": 1,
      "Chinese": 4,
  }
  if not higher_task_difficulty:
    task_difficulty_mapper = {
        "multiblimp": 1,
        "massive": 1,
        "xstorycloze": 1,
        "belebele": 2,
        "include": 2,
        "globalmmlulite": 2,
    }
  else:
    task_difficulty_mapper = {
        "multiblimp": 1,
        "massive": 2,
        "xstorycloze": 2,
        "belebele": 3,
        "include": 4,
        "globalmmlulite": 4,
    }
  spearman_mapper = dict()
  spearman_mapper.update(base_model_mapper)
  spearman_mapper.update(quantization_technique_mapper)
  spearman_mapper.update(bit_width_mapper)
  spearman_mapper.update(calibration_language_mapper)
  spearman_mapper.update(task_difficulty_mapper)

  # Figure for Line Plots Instances
  import plotly.graph_objects as go
  fig = go.Figure()


  if not exclude_model:
    columns = ["Base Model", "Quantization Technique", "Bit Width", "Calibration Language", "Task Difficulty", "Direct", "Cross", "Delta"]
  else:
    columns = ["Quantization Technique", "Bit Width", "Calibration Language", "Task Difficulty", "Direct", "Cross", "Delta"]
  df_chief = pd.DataFrame([], columns=columns)

  # Chief loop of every tasks
  task_num = len(list_eval_dataset)

  for i in range(task_num):
    eval_languages = list_eval_languages[i]
    runs = list_runs[i]
    base_models = list_base_models[i]
    eval_dataset = list_eval_dataset[i]
    print(f"Processing ... {eval_dataset}")

    eval_languages = list(sorted(eval_languages, key=lambda x: language_sort.get(x, 100)))
    columns = ["Base Model", "Quantization Technique", "Bit Width", "Calibration Language", "Task Difficulty"]
    columns.extend(eval_languages)

    data_acc = []

    # Iterate over the runs and print details
    for run in runs:
      config = run.config
      summary = run.summary

      if config["quantization_technique"] == "sinq":
        config["calibration_language"] = None
        continue

      if config["calibration_language"] == "WuuChinese":
        continue
      if config["quantization_technique"] == "Unquantized":
        continue

      run_data_acc = []
      run_data_acc.extend([run.config["base_model"], config["quantization_technique"], config["bit_width"], config["calibration_language"], eval_dataset.lower()])
      for lang in eval_languages:
        run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_acc_{lang}", None))

      data_acc.append(run_data_acc)

    df_acc_chief = pd.DataFrame(data_acc, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language', "Bit Width"], key=lambda x: x.map(custom_sort))

    # Agg Eng Eval on include
    if eval_dataset.lower() == "include" and not compile and 'telugu' not in eval_languages:
      for df_acc in [df_acc_chief]:
        df_acc.insert(5, 'english', (df_acc['german'] + df_acc['dutch']) / (2))
        df_acc.drop(["german", "dutch"], axis=1, inplace=True)
        replace_index = min(eval_languages.index("german"), eval_languages.index("dutch"))
        eval_languages.insert(replace_index, "english")
        eval_languages.remove("german")
        eval_languages.remove("dutch")

    # Calculate Direct, Cross, and Stats
    ## Groupby's apply is applied to each group's dataframe
    df_acc_chief = df_acc_chief.groupby(['Base Model', 'Quantization Technique', "Bit Width", "Task Difficulty", "Calibration Language"], as_index=False, group_keys=True).agg("mean").reset_index(drop=True)
    df_acc_chief_groupby = df_acc_chief.groupby(['Base Model', 'Quantization Technique', "Bit Width", "Task Difficulty"], as_index=False, group_keys=True)
    # df_acc_chief_groupby.apply(print)
    # Calib Language Series = x["Calibration Language"].apply(reverse_eval_label[eval_dataset.lower()]) -> Transform each Calib into eval dataset lang's name
    # display(df_acc_chief_groupby.apply(lambda x: x.loc[x["Calibration Language"], x["Calibration Language"].apply(reverse_eval_label[eval_dataset.lower()])]))
    # df_acc_chief_groupby.apply(lambda x: print(x["Calibration Language"].map(reverse_eval_label[eval_dataset.lower()]).to_list()))
    # df_acc_chief_groupby.apply(lambda x: print(x.index.to_list()))
    # df_acc_chief_stats = pd.DataFrame([], columns=["Direct", "Cross", "Count"])
    # df_acc_chief_groupby.apply(lambda x: print([x.loc[i, j] for i, j in zip(x.index.to_list(), x["Calibration Language"].map(reverse_eval_label[eval_dataset.lower()]).to_list())]))
    # display(df_acc_chief_groupby.apply(lambda x: x.loc[:, x["Calibration Language"].map(reverse_eval_label[eval_dataset.lower()])].sum(axis=0)))

    df_acc_chief["Direct"] = df_acc_chief_groupby.apply(lambda x: pd.Series([x.loc[i, j] for i, j in zip(x.index.to_list(), x["Calibration Language"].map(reverse_eval_label[eval_dataset.lower()]).to_list())], index=x.index)).reset_index(level=0, drop=True)
    df_acc_chief["Cross"] = df_acc_chief_groupby.apply(lambda x: x.loc[:, x["Calibration Language"].map(reverse_eval_label[eval_dataset.lower()])].sum(axis=0).set_axis(x.index)).reset_index(level=0, drop=True)
    df_acc_chief["Count"] = df_acc_chief_groupby.apply(lambda x: pd.Series(x["Calibration Language"].size, index=x.index)).reset_index(level=0, drop=True)
    # Move 'Target' column to the first position (index 0)
    # count = df_acc_chief.pop('Count')
    # df_acc_chief.insert(0, 'Count', count)
    #   {
    #     "Direct": df_acc_chief_groupby.apply(lambda x: [x.loc[i, j] for i, j in zip(x.index.to_list(), x["Calibration Language"].map(reverse_eval_label[eval_dataset.lower()]).to_list())]),
    #     "Cross": df_acc_chief_groupby.apply(lambda x: x.loc[x["Calibration Language"].apply(reverse_eval_label[eval_dataset.lower()])].sum()),
    #     "Count": df_acc_chief_groupby.apply(lambda x: x["Calibration Language"].count()),
    # })

    df_acc_chief["Cross"] = (df_acc_chief["Cross"] - df_acc_chief["Direct"]) / (df_acc_chief["Count"] - 1)
    df_acc_chief["Delta"] = df_acc_chief["Direct"] - df_acc_chief["Cross"]

    # Drop the unused columns later in spearman
    # df_acc_chief = df_acc_chief.drop("Random Seed", axis=1)
    df_acc_chief = df_acc_chief.drop("Count", axis=1)
    if exclude_model:
      df_acc_chief = df_acc_chief.drop("Base Model", axis=1)
    df_acc_chief = df_acc_chief.drop(eval_languages, axis=1)

    # Convert all columns into numeric even if ordinal
    df_acc_chief = df_acc_chief.map(lambda x: spearman_mapper.get(x, x))

    # Concat back to the chief
    df_chief = pd.concat([df_chief, df_acc_chief], axis=0)

  display(df_chief)
  display(df_chief.corr(method=corr_method).style.background_gradient(cmap='coolwarm').format(precision=2))

  def insert_and_return(df, loc, column, value):
    df_copy = df.copy()
    value.columns = [column]
    value.index = df.index
    df_copy = pd.concat([value, df_copy], ignore_index=True, axis=1)
    # df_copy.insert(loc, column, value, allow_duplicates=True)
    # df_copy.T.insert(loc, column, value, allow_duplicates=True)
    return df_copy

  def display_df_cor(df, filter, value):
    if filter:
      return df[df[filter] == value].corr(method=corr_method).drop(filter, axis=0).drop(filter, axis=1)
    else:
      return df.corr(method=corr_method)

  def count_df(df, filter, value):
    if filter:
      return len(df[df[filter] == value])
    else:
      return len(df)

  def display_plotly(df, pdf_name=''):
    axis_mapper = {
        "Quantization Technique": "Quant Tech",
        "Calibration Language": "Lang Resource",
        "Delta": "Delta Acc",
        "Base Model": "Model's Release"
    }
    if not display_raw_score:
      df = df.drop(["Direct", "Cross"], axis=0).drop(["Direct", "Cross"], axis=1)

    # Reorient to top and left, rather than bottom and left
    df = df.reindex(index=df.index[::-1])

    if truncate_heatmap:
      df = df.drop(["Base Model", "Quantization Technique", "Bit Width", "Calibration Language", "Task Difficulty"], axis=0, errors='ignore')


    fig = go.Figure(data=go.Heatmap(
        z=df.values,
        x=df.columns.map(lambda x: axis_mapper.get(x, x)),
        y=df.index.map(lambda x: axis_mapper.get(x, x)),
        text=df.values,
        texttemplate="%{text:.2f}",
        # text_auto=True,
        zmin=-1.0, zmax=1.0,
        colorscale ="RdBu"
    ))

    fig.update_xaxes(automargin=False)
    fig.update_yaxes(automargin=False)

    fig.update_layout(
        # title_text=f"{base_model} {custom_title} {eval_dataset} (Max 1.0, +other language improve): Delta acc with Unquantized of <b>{quantization_technique} {bit_width}-bit</b> model",
        # xaxis_title_text='Quantization Technique - Calibration Language',
        # yaxis_title_text='Base Model - Eval Language',
        margin=dict(t=60, b=15 if write_pdf else 0, l=100 if not truncate_heatmap else 60, r=0)
    )

    fig.layout.height = 60 + len(df.index) * 50 + (15 if write_pdf else 0)
    fig.layout.width = (100 if not truncate_heatmap else 60) + len(df.columns) * 50 + 0

    fig.update_xaxes(side="top")
    fig.update_traces(showscale=False)

    if write_pdf:
      fig.write_image(pdf_name)
    fig.show()

  def display_plotly_multi(dfs, controls, filter, counts=None, pdf_name=''):
    axis_mapper = {
        "Quantization Technique": "Quant Tech",
        "Calibration Language": "Lang Resource",
        "Delta": "Delta Acc",
        "Base Model": "Model's Release"
    }
    if not display_raw_score:
      for df in dfs:
        df.drop(["Direct", "Cross"], axis=0, inplace=True)
        df.drop(["Direct", "Cross"], axis=1, inplace=True)
    if truncate_heatmap:
      for df in dfs:
        df.drop(["Base Model", "Quantization Technique", "Bit Width", "Calibration Language", "Task Difficulty"], axis=0, errors='ignore', inplace=True)

    # Reorient to top and left, rather than bottom and left
    df = pd.concat(dfs, axis=0, ignore_index=True)
    df.index = controls
    if counts:
      df.insert(0, "Count", counts)
    df = df.reindex(index=df.index[::-1])
    display(df)
    text = df.map(lambda x: f"{x:.2f}" if x <= 1 else f"{x:d}")
    fig = go.Figure(data=go.Heatmap(
        z=df.assign(Count=[0] * len(counts)).values if counts else df.values,
        x=df.columns.map(lambda x: axis_mapper.get(x, x)),
        y=df.index.map(lambda x: axis_mapper.get(x, x)),
        # text=df.values,
        text=text,
        # texttemplate="%{text:.2f}",
        texttemplate="%{text}",
        # text_auto=True,
        zmin=-1.0, zmax=1.0,
        colorscale ="RdBu"
    ))

    fig.update_xaxes(automargin=False)
    fig.update_yaxes(automargin=False)

    fig.update_layout(
        # title_text=f"{base_model} {custom_title} {eval_dataset} (Max 1.0, +other language improve): Delta acc with Unquantized of <b>{quantization_technique} {bit_width}-bit</b> model",
        # xaxis_title_text='Quantization Technique - Calibration Language',
        yaxis_title_text=f'Control Variable<br>{filter}',
        margin=dict(t=60, b=15 if write_pdf else 0, l=120 if not truncate_heatmap else 80 + (25 if filter == "Calibration Language" else 0), r=0)
    )

    fig.layout.height = 60 + len(df.index) * 50 + (15 if write_pdf else 0)
    fig.layout.width = (120 if not truncate_heatmap else 80) + (25 if filter == "Calibration Language" else 0) + len(df.columns) * 50 + 0

    fig.update_xaxes(side="top")
    fig.update_traces(showscale=False)

    if write_pdf:
      fig.write_image(pdf_name)
    fig.show()

  #################### Start
  print("General")
  display_plotly(display_df_cor(df_chief, "", None), pdf_name="5lang_correlation_general.pdf")

  display(df_chief)
  print("Bit Width")
  print(df_chief["Bit Width"].value_counts())
  counts = []
  dfs = []
  dfs.append(display_df_cor(df_chief, "Bit Width", 1))
  counts.append(count_df(df_chief, "Bit Width", 1))
  dfs.append(display_df_cor(df_chief, "Bit Width", 2))
  counts.append(count_df(df_chief, "Bit Width", 2))
  if include_4bit:
    dfs.append(display_df_cor(df_chief, "Bit Width", 3))
    counts.append(count_df(df_chief, "Bit Width", 3))
    display_plotly_multi(dfs, ["2-bit", "3-bit", "4-bit"], "Bit Width", counts=counts, pdf_name="5lang_correlation_control_bit.pdf")
  else:
    display_plotly_multi(dfs, ["2-bit", "3-bit"], "Bit Width", counts=counts, pdf_name="5lang_correlation_control_bit.pdf")

  print("Quantization Technique")
  print(df_chief["Quantization Technique"].value_counts())
  counts = []
  dfs = []
  dfs.append(display_df_cor(df_chief, "Quantization Technique", 1))
  counts.append(count_df(df_chief, "Quantization Technique", 1))
  dfs.append(display_df_cor(df_chief, "Quantization Technique", 2))
  counts.append(count_df(df_chief, "Quantization Technique", 2))
  dfs.append(display_df_cor(df_chief, "Quantization Technique", 3))
  counts.append(count_df(df_chief, "Quantization Technique", 3))
  display_plotly_multi(dfs, ["GPTQ", "SliM", "TaCQ"], "Quantization Technique", counts=counts, pdf_name="5lang_correlation_control_quant.pdf")

  print("Calibration Language")
  print(df_chief["Calibration Language"].value_counts())
  counts = []
  dfs = []
  dfs.append(display_df_cor(df_chief, "Calibration Language", 1))
  counts.append(count_df(df_chief, "Calibration Language", 1))
  dfs.append(display_df_cor(df_chief, "Calibration Language", 2))
  counts.append(count_df(df_chief, "Calibration Language", 2))
  dfs.append(display_df_cor(df_chief, "Calibration Language", 3))
  counts.append(count_df(df_chief, "Calibration Language", 3))
  dfs.append(display_df_cor(df_chief, "Calibration Language", 4))
  counts.append(count_df(df_chief, "Calibration Language", 4))
  dfs.append(display_df_cor(df_chief, "Calibration Language", 5))
  counts.append(count_df(df_chief, "Calibration Language", 5))
  display_plotly_multi(dfs, ["Swahili", "Tamil", "Indonesian", "Chinese", "English"], "Calibration Language", counts=counts, pdf_name="5lang_correlation_control_calib.pdf")

  print("Task Difficulty")
  print(df_chief['Task Difficulty'].value_counts())
  if not higher_task_difficulty:
    counts = []
    dfs = []
    dfs.append(display_df_cor(df_chief, "Task Difficulty", 1))
    counts.append(count_df(df_chief, "Task Difficulty", 1))
    dfs.append(display_df_cor(df_chief, "Task Difficulty", 2))
    counts.append(count_df(df_chief, "Task Difficulty", 2))
    display_plotly_multi(dfs, ["Easy", "Hard"], "Task Difficulty", counts=counts, pdf_name="5lang_correlation_control_task.pdf")
  else:
    counts = []
    dfs = []
    for i in range(4):
      dfs.append(display_df_cor(df_chief, "Task Difficulty", i+1))
      counts.append(count_df(df_chief, "Task Difficulty", i+1))
    display_plotly_multi(dfs, ["MultiBlimp", "XStoryCloze & Massive", "Belebele", "GlobalMMLULite & INCLUDE"], "Task Difficulty", counts=counts, pdf_name="5lang_correlation_control_task.pdf")

In [ ]:
# Initialize the W&B API
api = Api()

# Parameters
list_eval_languages = []
list_runs = []
list_base_models = []
list_eval_dataset = []
list_bits = []

# MultiBlimp
eval_languages = ["eng", "tam", "zhoblimp"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "MultiBlimp"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.calibration_language": "Indonesian"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                         { "$and": [
                            {"config.quantization_technique": "sinq"},
                            {"$not": {"config.bit_width": "4-bit"}},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"$not": {"config.base_model": "Qwen/Qwen3-14B"}},
                            # {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$gt": '2026-07-01'}},
                          ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Massive Scenario
eval_languages = ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN']
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
eval_dataset = "Massive"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario-2000"},
                            ]
                          },
                          # {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                         { "$and": [
                            {"config.quantization_technique": "sinq"},
                            {"$not": {"config.bit_width": "4-bit"}},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"$not": {"config.base_model": "Qwen/Qwen3-14B"}},
                            # {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$gt": '2026-07-01'}},
                          ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# XStoryCloze
eval_languages = ["en", "id", "zh"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
eval_dataset = "XStoryCloze"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                         { "$and": [
                            {"config.quantization_technique": "sinq"},
                            {"$not": {"config.bit_width": "4-bit"}},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"$not": {"config.base_model": "Qwen/Qwen3-14B"}},
                            # {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$gt": '2026-07-01'}},
                          ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Belebele
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "Belebele"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                         { "$and": [
                            {"config.quantization_technique": "sinq"},
                            {"$not": {"config.bit_width": "4-bit"}},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"$not": {"config.base_model": "Qwen/Qwen3-14B"}},
                            # {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$gt": '2026-07-01'}},
                          ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Global MMLU Lite
eval_languages = ["en", "zh", "id", "sw"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "GlobalMMLULite"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                         { "$and": [
                            {"config.quantization_technique": "sinq"},
                            {"$not": {"config.bit_width": "4-bit"}},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"$not": {"config.base_model": "Qwen/Qwen3-14B"}},
                            # {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$gt": '2026-07-01'}},
                          ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# INCLUDE
eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Tamil"]
eval_languages = [eval_lang.lower() for eval_lang in eval_languages]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "INCLUDE"
bits = ["2-bit", "3-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          # {"config.evaluation_dataset": eval_dataset.lower()},
                          { "$or":
                            [
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]
                          },
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]
                        },
                         { "$and": [
                            {"config.quantization_technique": "sinq"},
                            {"$not": {"config.bit_width": "4-bit"}},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"$not": {"config.base_model": "Qwen/Qwen3-14B"}},
                            # {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$gt": '2026-07-01'}},
                          ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": '2026-07-01'}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

print(f"{list_eval_languages=}")
print(f"{list_runs=}")
print(f"{list_base_models=}")
print(f"{list_eval_dataset=}")
print(f"{list_bits=}")
print()

# Run the function
deliver_spearman_5langdegree(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, custom_title="", display_mean=False, compile=False, corr_method="spearman", display_raw_score=False, write_pdf=True, truncate_heatmap=True, exclude_model=True, higher_task_difficulty=True)

list_eval_languages=[['eng', 'tam', 'zhoblimp'], ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN'], ['en', 'id', 'zh'], ['eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn', 'zho_Hans'], ['en', 'zh', 'id', 'sw'], ['dutch', 'german', 'chinese', 'indonesian', 'tamil']]
list_runs=[<Runs rofifpriyo-/calibration-on-quantized-multilingual>, <Runs rofifpriyo-/calibration-on-quantized-multilingual>, <Runs rofifpriyo-/calibration-on-quantized-multilingual>, <Runs rofifpriyo-/calibration-on-quantized-multilingual>, <Runs rofifpriyo-/calibration-on-quantized-multilingual>, <Runs rofifpriyo-/calibration-on-quantized-multilingual>]
list_base_models=[['Qwen/Qwen3-8B', 'CohereLabs/aya-expanse-8b', 'meta-llama/Llama-3.1-8B-Instruct'], ['Qwen/Qwen3-8B', 'CohereLabs/aya-expanse-8b'], ['Qwen/Qwen3-8B', 'CohereLabs/aya-expanse-8b'], ['Qwen/Qwen3-8B', 'CohereLabs/aya-expanse-8b', 'meta-llama/Llama-3.1-8B-Instruct'], ['Qwen/Qwen3-8B', 'CohereLabs/aya-expanse-8b', 'meta-llama/Llama-3.1-8B-Instruct'], ['Qwen/Qwen3-8B'

/tmp/ipykernel_2750/2883402721.py:212: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_2750/2883402721.py:213: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_2750/2883402721.py:214: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Ei

Processing ... Massive


/tmp/ipykernel_2750/2883402721.py:212: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_2750/2883402721.py:213: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_2750/2883402721.py:214: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Ei

Processing ... XStoryCloze


...

,Count,Bit Width,Calibration Language,Task Difficulty,Delta
TaCQ,128,-0.269796,-0.267431,-0.228364,1.0
SliM,120,0.070247,-0.027890,-0.370031,1.0
GPTQ,128,0.299609,-0.196966,-0.191358,1.0


Calibration Language
Calibration Language
4    96
5    96
3    78
2    58
1    48
Name: count, dtype: int64


,Count,Quantization Technique,Bit Width,Task Difficulty,Delta
English,96,0.156069,-0.325529,-0.444899,1.0
Chinese,96,-0.074582,-0.029320,-0.140247,1.0
Indonesian,78,0.137738,0.026194,-0.045970,1.0
Tamil,58,0.169092,0.256448,-0.438460,1.0
Swahili,48,0.101303,0.461692,-0.326469,1.0


Task Difficulty
Task Difficulty
4    142
2     94
3     88
1     52
Name: count, dtype: int64


,Count,Quantization Technique,Bit Width,Calibration Language,Delta
GlobalMMLULite & INCLUDE,142,0.052968,0.109266,-0.190050,1.0
Belebele,88,0.086955,0.393675,-0.064209,1.0
XStoryCloze & Massive,94,0.186498,-0.337570,-0.254319,1.0
MultiBlimp,52,0.038499,-0.340836,-0.363254,1.0


###### Include 4-bit

In [ ]:
# Initialize the W&B API
api = Api()

# Parameters
list_eval_languages = []
list_runs = []
list_base_models = []
list_eval_dataset = []
list_bits = []

# MultiBlimp
eval_languages = ["eng", "tam", "zhoblimp"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "MultiBlimp"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          {"$not": {"config.calibration_language": "Indonesian"}},
                          {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"created_at": {"$lt": "2026-03-01"}}]}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Massive Scenario
eval_languages = ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN']
num_samples = 128
base_models = ["Qwen/Qwen3-8B"]
eval_dataset = "Massive"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": f"{eval_dataset.lower()}-scenario"},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# XStoryCloze
eval_languages = ["en", "id", "zh"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]
eval_dataset = "XStoryCloze"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Belebele
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "Belebele"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                            {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.enable_thinking": False},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# Global MMLU Lite
eval_languages = ["en", "zh", "id", "sw"]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "GlobalMMLULite"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Tamil"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

# INCLUDE
eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Tamil"]
eval_languages = [eval_lang.lower() for eval_lang in eval_languages]
num_samples = 128
base_models = ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"]
eval_dataset = "INCLUDE"
bits = ["2-bit", "3-bit", "4-bit"]

runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                filters={"$or": [
                        { "$and": [
                          {"config.num_calibration_samples": num_samples},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"config.output_type": "multiple_choice"},
                          {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          {"$not": {"config.calibration_language": "WuuChinese"}},
                          {"$not": {"config.calibration_language": "Swahili"}},
                          # {"$not": {"config.bit_width": "4-bit"}},
                          {"$not": {"config.bit_width": "8-bit"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]
                        },
                        { "$and" : [
                          {"config.quantization_technique": "Unquantized"},
                          {"config.evaluation_dataset": eval_dataset.lower()},
                          {"config.enable_thinking": False},
                          {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                          {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                        ]}
                    ]}
                )

list_eval_languages.append(eval_languages)
list_runs.append(runs)
list_base_models.append(base_models)
list_eval_dataset.append(eval_dataset)
list_bits.append(bits)

print(f"{list_eval_languages=}")
print(f"{list_runs=}")
print(f"{list_base_models=}")
print(f"{list_eval_dataset=}")
print(f"{list_bits=}")
print()

# Run the function
deliver_spearman_5langdegree(list_eval_languages=list_eval_languages, list_runs=list_runs, list_base_models=list_base_models, list_eval_dataset=list_eval_dataset, bits=list_bits, custom_title="", display_mean=False, compile=False, corr_method="spearman", display_raw_score=False, write_pdf=True, truncate_heatmap=True, include_4bit=True)

list_eval_languages=[['eng', 'tam', 'zhoblimp'], ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN'], ['en', 'id', 'zh'], ['eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn', 'zho_Hans'], ['en', 'zh', 'id', 'sw'], ['dutch', 'german', 'chinese', 'indonesian', 'tamil']]
list_runs=[<Runs rofifpriyo-/calibration-on-quantized-multilingual>, <Runs rofifpriyo-/calibration-on-quantized-multilingual>, <Runs rofifpriyo-/calibration-on-quantized-multilingual>, <Runs rofifpriyo-/calibration-on-quantized-multilingual>, <Runs rofifpriyo-/calibration-on-quantized-multilingual>, <Runs rofifpriyo-/calibration-on-quantized-multilingual>]
list_base_models=[['Qwen/Qwen3-8B', 'CohereLabs/aya-expanse-8b', 'meta-llama/Llama-3.1-8B-Instruct'], ['Qwen/Qwen3-8B'], ['Qwen/Qwen3-8B', 'CohereLabs/aya-expanse-8b'], ['Qwen/Qwen3-8B', 'CohereLabs/aya-expanse-8b', 'meta-llama/Llama-3.1-8B-Instruct'], ['Qwen/Qwen3-8B', 'CohereLabs/aya-expanse-8b', 'meta-llama/Llama-3.1-8B-Instruct'], ['Qwen/Qwen3-8B', 'CohereLabs/aya-expanse-8b'

/tmp/ipykernel_2760/606675486.py:193: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_2760/606675486.py:194: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_2760/606675486.py:195: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Eithe

Processing ... Massive


/tmp/ipykernel_2760/606675486.py:193: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_2760/606675486.py:194: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_2760/606675486.py:195: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Eithe

Processing ... XStoryCloze


...

General


Bit Width


Quantization Technique


Calibration Language


Task Difficulty
